# **Knowledge Graph 구현**

---

## 학습 목표

- 정형 데이터(CSV)를 Knowledge Graph로 변환하는 방법 이해
- Neo4j 제약조건(UNIQUE, NOT NULL)과 인덱스 활용
- Cypher 쿼리로 그래프 데이터 조회 및 분석
- Full-text 인덱스와 벡터 인덱스를 활용한 고급 검색 구현
- LangChain과 Neo4j를 통합하여 Text-to-Cypher 시스템 구축

## 사전 준비

- Neo4j Desktop 5.x 설치 및 APOC 플러그인 활성화
- Python 환경: langchain-neo4j, langchain-openai, pandas 설치
- OpenAI API 키 (임베딩 및 LLM 사용)

---

## 1. Neo4J Desktop 환경 설정

- 버전 1.6.2 설치: https://neo4j.com/deployment-center/

- Neo4J Desktop 설치 후, Neo4J Desktop에서 새로운 프로젝트 생성
    - 프로젝트 이름: `test`
    - 데이터베이스 버전: `5.24.0`

- 데이터베이스에서 플러그인 설치
    - `APOC`

- 데이터베이스 Settings에서 다음 설정 추가 (apoc 검색해서 기존 설정에 추가)
    - `dbms.security.procedures.unrestricted=jwt.security.*,apoc.*,apoc.meta.*`

- Neo4J Desktop 사용(.env)
    ```
    NEO4J_URI=bolt://localhost:7687
    NEO4J_USERNAME=neo4j
    NEO4J_PASSWORD=modulab1234
    NEO4J_DATABASE=neo4j
    ```

`(1) Env 환경변수`

In [3]:
from dotenv import load_dotenv
load_dotenv()

True

`(2) 라이브러리`

In [4]:
import os
from glob import glob

from pprint import pprint
import json

import numpy as np
import pandas as pd

import warnings
warnings.filterwarnings('ignore')

`(3) Neo4j 설정`

In [5]:
import os
from langchain_neo4j import Neo4jGraph

NEO4J_URI = os.getenv("NEO4J_URI")
NEO4J_USERNAME = os.getenv("NEO4J_USERNAME")
NEO4J_PASSWORD = os.getenv("NEO4J_PASSWORD")
NEO4J_DATABASE = os.getenv("NEO4J_DATABASE")

print(f"uri: {NEO4J_URI}")
print(f"username: {NEO4J_USERNAME}")
print(f"password: {NEO4J_PASSWORD}")
print(f"database: {NEO4J_DATABASE}")
# LangChain 도구 활용 - DB 연결 객체 초기화 
graph = Neo4jGraph(  
    url=NEO4J_URI, 
    username=NEO4J_USERNAME, 
    password=NEO4J_PASSWORD,
    database=NEO4J_DATABASE,
    enhanced_schema=True,
    )

graph.query("MATCH (n) RETURN n LIMIT 5;")

uri: neo4j://127.0.0.1:7687
username: neo4j
password: modulab1234
database: neo4j


[]

`(4) 기존 DB의 모든 내용 삭제`

In [6]:
def reset_database(graph):
    """
    데이터베이스 초기화하기
    """
    # 모든 노드와 관계 삭제
    graph.query("MATCH (n) DETACH DELETE n")
    
    # 모든 제약조건 삭제
    constraints = graph.query("SHOW CONSTRAINTS")
    for constraint in constraints:
        constraint_name = constraint.get("name")
        if constraint_name:
            graph.query(f"DROP CONSTRAINT {constraint_name}")
    
    # 모든 인덱스 삭제
    indexes = graph.query("SHOW INDEXES")
    for index in indexes:
        index_name = index.get("name")
        index_type = index.get("type")
        if index_name and index_type != "CONSTRAINT":
            graph.query(f"DROP INDEX {index_name}")
    
    print("데이터베이스가 초기화되었습니다.")

# 데이터베이스 초기화
reset_database(graph)

데이터베이스가 초기화되었습니다.


In [7]:
# 그래프 스키마 조회
graph.refresh_schema()
print(graph.schema)

Node properties:

Relationship properties:

The relationships:



## 2. 정형 데이터를 KG로 변환

> 💡 **정형 CSV → KG 변환의 5단계 파이프라인**
>
```mermaid
flowchart LR
    CSV[(📄 CSV 파일)] --> DF[pandas DataFrame<br/>로드 & 정제]
    DF --> CON[제약조건 설정<br/>UNIQUE / NOT NULL]
    CON --> NODE[노드 MERGE<br/>ETF, Company, Category]
    NODE --> REL[관계 MERGE<br/>-MANAGED_BY-, -CATEGORIZED_AS-]
    REL --> IDX[인덱스 생성<br/>B-tree / Fulltext / Vector]
    style CON fill:#fff4e1
    style IDX fill:#e8f5e9
```

> 🔑 **순서가 중요합니다**: 
> **제약조건 → 노드 → 관계 → 인덱스**. 제약조건을 먼저 설정해야 
> 중복 노드가 원천 차단되고, 인덱스는 데이터가 전부 들어간 뒤 만들어야 빌드 시간이 짧아집니다.


`(1) Load CSV Data`

- etf info 데이터 활용 (data/etf_list.csv)

In [8]:
# CSV 파일 읽기
df = pd.read_csv('data/etf_list.csv', encoding='cp949')

df.shape

(930, 14)

In [9]:
df.head()

,종목코드,종목명,상장일,분류체계,운용사,수익률(최근 1년),기초지수,추적오차,순자산총액,괴리율,변동성,복제방법,총보수,과세유형
0,466400,1Q 25-08 회사채(A+이상)액티브,2023/09/19,채권-회사채-단기,하나자산운용,4.52,KIS 2025-08만기형 크레딧 A+이상 지수(총수익),0.11,111916276404,0.03,매우낮음,실물(액티브),0.10,배당소득세(보유기간과세)
1,491610,1Q CD금리액티브(합성),2024/09/24,기타,하나자산운용,0.00,KIS 하나 CD금리 총수익지수,0.05,316206006696,0.02,매우낮음,합성(액티브),0.02,배당소득세(보유기간과세)
2,451060,1Q K200액티브,2023/01/31,주식-시장대표,하나자산운용,-3.66,코스피 200,0.77,99754348820,-0.01,높음,실물(액티브),0.18,배당소득세(보유기간과세)
3,463290,1Q 단기금융채액티브,2023/08/03,채권-혼합-단기,하나자산운용,4.01,MK 머니마켓 지수(총수익),0.05,252717462257,0.00,매우낮음,실물(액티브),0.08,배당소득세(보유기간과세)
4,479080,1Q 머니마켓액티브,2024/04/02,채권-혼합-단기,하나자산운용,0.00,KIS-하나 MMF 지수(총수익),0.06,308255065986,-0.01,매우낮음,실물(액티브),0.05,배당소득세(보유기간과세)


`(2) 제약 조건 생성`

In [10]:
# 종목코드(code)는 ETF마다 고유하므로 유일성 제약조건을 설정 
unique_code_constraint = """
CREATE CONSTRAINT etf_code_unique IF NOT EXISTS
FOR (e:ETF) 
REQUIRE e.code IS UNIQUE
"""
graph.query(unique_code_constraint)

[]

In [11]:
graph.query("SHOW CONSTRAINTS")

[{'id': 3,
  'name': 'etf_code_unique',
  'type': 'UNIQUENESS',
  'entityType': 'NODE',
  'labelsOrTypes': ['ETF'],
  'properties': ['code'],
  'ownedIndex': 'etf_code_unique',
  'propertyType': None}]

In [12]:
# 종목코드와 종목명은 반드시 존재해야 하는 필수 속성으로 설정 

exists_code_constraint = """
// 종목코드가 반드시 존재해야 하는 필수 속성으로 설정
CREATE CONSTRAINT etf_code_exists IF NOT EXISTS
FOR (e:ETF)
REQUIRE e.code IS NOT NULL
"""
graph.query(exists_code_constraint)

exists_name_constraint = """
// 종목명이 반드시 존재해야 하는 필수 속성으로 설정
CREATE CONSTRAINT etf_name_exists IF NOT EXISTS
FOR (e:ETF)
REQUIRE e.name IS NOT NULL
"""
graph.query(exists_name_constraint)

[]

In [13]:
graph.query("SHOW INDEXES")

[{'id': 2,
  'name': 'etf_code_unique',
  'state': 'ONLINE',
  'populationPercent': 100.0,
  'type': 'RANGE',
  'entityType': 'NODE',
  'labelsOrTypes': ['ETF'],
  'properties': ['code'],
  'indexProvider': 'range-1.0',
  'owningConstraint': 'etf_code_unique',
  'lastRead': None,
  'readCount': None}]

`(3) Neo4J 그래프 DB에서 CSV파일 업로드`

In [14]:
df.head(1)

,종목코드,종목명,상장일,분류체계,운용사,수익률(최근 1년),기초지수,추적오차,순자산총액,괴리율,변동성,복제방법,총보수,과세유형
0,466400,1Q 25-08 회사채(A+이상)액티브,2023/09/19,채권-회사채-단기,하나자산운용,4.52,KIS 2025-08만기형 크레딧 A+이상 지수(총수익),0.11,111916276404,0.03,매우낮음,실물(액티브),0.1,배당소득세(보유기간과세)


In [15]:
# pandas의 to_dict를 활용한 데이터 변환
def clean_value(value):
    """NaN 값을 None으로 변환하는 헬퍼 함수"""
    if pd.isna(value):
        return None
    if isinstance(value, (int, float)) and pd.notna(value):
        return value
    return value

# DataFrame을 딕셔너리 리스트로 변환
etf_data = []
for record in df.to_dict('records'):
    cleaned_record = {k: clean_value(v) for k, v in record.items()}

    # 컬럼명을 영어로 매핑
    etf_record = {
        'code': cleaned_record['종목코드'],
        'name': cleaned_record['종목명'],
        'listingDate': cleaned_record['상장일'],
        'category': cleaned_record['분류체계'],
        'company': cleaned_record['운용사'],
        'yearReturn': cleaned_record['수익률(최근 1년)'],
        'baseIndex': cleaned_record['기초지수'],
        'trackingError': cleaned_record['추적오차'],
        'netAsset': cleaned_record['순자산총액'],
        'disparityRatio': cleaned_record['괴리율'],
        'volatility': cleaned_record['변동성'],
        'replicationMethod': cleaned_record['복제방법'],
        'totalFee': cleaned_record['총보수'],
        'taxType': cleaned_record['과세유형']
    }
    etf_data.append(etf_record)

# 배치 처리 실행
BATCH_SIZE = 100
batch_query = """
UNWIND $etf_list AS etf
MERGE (etf_node:ETF {code: etf.code})
SET etf_node += {
    name: etf.name,
    listingDate: etf.listingDate,
    category: etf.category,
    company: etf.company,
    yearReturn: etf.yearReturn,
    baseIndex: etf.baseIndex,
    trackingError: etf.trackingError,
    netAsset: etf.netAsset,
    disparityRatio: etf.disparityRatio,
    volatility: etf.volatility,
    replicationMethod: etf.replicationMethod,
    totalFee: etf.totalFee,
    taxType: etf.taxType
}
"""

from tqdm import tqdm

# 배치 처리
for i in tqdm(range(0, len(etf_data), BATCH_SIZE)):
    batch = etf_data[i:i + BATCH_SIZE]
    graph.query(batch_query, params={'etf_list': batch})

print("ETF 데이터 배치 로드 완료")

100%|██████████| 10/10 [00:02<00:00,  4.81it/s]

ETF 데이터 배치 로드 완료


In [16]:
# ETF 노드 개수 확인
count_query = """
MATCH (e:ETF)   // 모든 ETF 노드 검색
RETURN count(e) AS count   // 노드 개수 반환
"""
graph.query(count_query)

[{'count': 930}]

In [17]:
# ETF 노드 속성으로 조건을 주고 검색
query = """
MATCH (e:ETF {code: '451060'})   // 종목코드가 451060인 ETF 노드 검색
RETURN e   // 노드 반환
"""
graph.query(query)

[{'e': {'trackingError': 0.77,
   'netAsset': 99754348820,
   'replicationMethod': '실물(액티브)',
   'listingDate': '2023/01/31',
   'code': '451060',
   'baseIndex': '코스피 200',
   'volatility': '높음',
   'yearReturn': -3.66,
   'disparityRatio': -0.01,
   'totalFee': 0.18,
   'name': '1Q K200액티브',
   'company': '하나자산운용',
   'category': '주식-시장대표',
   'taxType': '배당소득세(보유기간과세)'}}]

In [18]:
# 운용사(Company) 노드 생성 및 관계 설정
company_query = """
MATCH (e:ETF)       // ETF 노드에서
WITH DISTINCT e.company AS companyName    // 회사 이름을 가져옴
WHERE companyName IS NOT NULL             // 회사 이름이 NULL이 아닌 경우
MERGE (c:Company {name: companyName})     // Company 노드 생성
RETURN count(c) AS company_count          // 생성된 Company 노드 개수 반환
"""

result = graph.query(company_query)
print(result)

[{'company_count': 26}]


In [19]:
# ETF와 운용사 간의 관계 생성
relationship_query = """
MATCH (e:ETF), (c:Company)     // ETF와 Company 노드 모두 선택
WHERE e.company = c.name       // 회사 이름이 일치하는 경우
MERGE (c)-[r:MANAGES]->(e)      // Company 노드에서 ETF 노드로 MANAGES 관계 생성
RETURN count(r) AS relationship_count   // 생성된 관계 개수 반환
"""
graph.query(relationship_query)

[{'relationship_count': 930}]

`(4) 관계 추가`

In [20]:
# 분류체계(Category) 유일성 제약조건 설정
unique_category_constraint = """
CREATE CONSTRAINT category_name_unique IF NOT EXISTS
FOR (c:Category)
REQUIRE c.name IS UNIQUE
"""
graph.query(unique_category_constraint)

[]

In [21]:
# 분류체계(Category) 노드 생성
category_query = """
MATCH (e:ETF)           // ETF 노드에서
WITH DISTINCT e.category AS categoryName    // 분류체계 이름을 가져옴
WHERE categoryName IS NOT NULL              // 분류체계 이름이 NULL이 아닌 경우
MERGE (c:Category {name: categoryName})     // Category 노드 생성
RETURN count(c) as CategoryCount            // 생성된 Category 노드 수 반환
"""
graph.query(category_query)

[{'CategoryCount': 51}]

In [22]:
# ETF와 분류체계(Category) 간의 관계 생성
relationship_query = """
MATCH (e:ETF), (c:Category)     // ETF와 Category 노드 모두 선택
WHERE e.category = c.name       // 분류체계 이름이 일치하는 경우
MERGE (e)-[r:BELONGS_TO]->(c)    // ETF 노드에서 Category 노드로 BELONGS_TO 관계 생성
RETURN count(r) as RelationshipCount   // 생성된 관계 수 반환
"""
graph.query(relationship_query)

[{'RelationshipCount': 930}]

In [23]:
# 특정 카테코리의 ETF 노드 조회
category_query = """
MATCH (c:Category {name: '채권-혼합-단기'})<-[:BELONGS_TO]-(etf:ETF)  // '채권-혼합-단기' 카테고리에 속하는 ETF 노드 검색
RETURN etf.name, etf.yearReturn, c.name   // ETF 노드의 이름과 수익률 반환, 카테고리 반환
ORDER BY etf.yearReturn DESC // 수익률 기준 내림차순 정렬
LIMIT 5  // 상위 5개 노드만 반환
"""

graph.query(category_query)

[{'etf.name': '히어로즈 25-09 미국채권(AA-이상)액티브',
  'etf.yearReturn': 13.48,
  'c.name': '채권-혼합-단기'},
 {'etf.name': 'RISE KP달러채권액티브', 'etf.yearReturn': 13.06, 'c.name': '채권-혼합-단기'},
 {'etf.name': '히어로즈 단기채권ESG액티브', 'etf.yearReturn': 4.46, 'c.name': '채권-혼합-단기'},
 {'etf.name': 'SOL 초단기채권액티브', 'etf.yearReturn': 4.08, 'c.name': '채권-혼합-단기'},
 {'etf.name': 'RISE 머니마켓액티브', 'etf.yearReturn': 4.06, 'c.name': '채권-혼합-단기'}]

In [24]:
# 카테코리별 평균 수익률 등의 통계 데이터를 계산
category_stats_query = """
MATCH (c:Category)<-[:BELONGS_TO]-(etf:ETF)  // 모든 카테고리와 그에 속하는 ETF 노드 검색
RETURN c.name AS category,      // 카테고리 이름
       COUNT(etf) AS etf_count,    // ETF 개수
       AVG(etf.yearReturn) AS avg_yearReturn,   // 평균 수익률
       SUM(etf.netAsset) AS total_netAsset,     // 총 순자산총액
       AVG(etf.trackingError) AS avg_trackingError   // 평균 추적오차
ORDER BY total_netAsset DESC // 평균 수익률 기준 내림차순 정렬  
"""

category_stats = graph.query(category_stats_query)

# 결과를 DataFrame으로 변환
category_stats_df = pd.DataFrame(category_stats)
category_stats_df.head(10)  # 상위 10개 카테고리 출력

,category,etf_count,avg_yearReturn,total_netAsset,avg_trackingError
0,주식-시장대표,219,10.335662,54542201581193,1.905753
1,기타,28,6.062500,30458800291049,0.344286
2,주식-업종섹터-업종테마,226,8.954823,21895495304087,3.429336
3,채권-혼합-단기,25,2.606000,11544903535948,0.151200
4,채권-국공채-장기,46,3.551304,8388509573240,1.022391
5,채권-혼합-중기,12,6.338333,7328213777260,0.420000
6,주식-전략-배당,32,10.369687,5973531895918,1.015625
7,채권-회사채-단기,23,3.189130,5799230072671,0.145217
8,주식-전략-구조화,24,0.440417,3219498638692,1.032917
9,주식-업종섹터,6,29.843333,3200641966315,1.328333


## 3. GraphCypherQAChain 활용

> 💡 **Text-to-Cypher 내부 동작**
>
```mermaid
sequenceDiagram
    participant U as 사용자
    participant C as GraphCypherQAChain
    participant L as LLM
    participant N as Neo4j
    U->>C: "전기차 ETF 상위 5개?"
    C->>L: 스키마 + 질문 전달<br/>(프롬프트 구성)
    L-->>C: Cypher 쿼리 생성<br/>MATCH (e:ETF)...
    C->>N: Cypher 실행
    N-->>C: 결과 row
    C->>L: 결과 + 질문 재전달<br/>(자연어 요약 요청)
    L-->>C: 최종 답변 문자열
    C-->>U: "TIGER 전기차, KODEX 전기차..."
```

> ⚠️ **왜 LLM 을 두 번 호출하나요?** 
> 1차는 **Cypher 생성**, 2차는 **결과를 사람 친화적인 문장으로 정리**하기 위해서입니다. 
> 비용이 2배이므로 대용량 서비스라면 2차 호출은 포맷팅 함수로 대체하는 것도 고려하세요.


In [25]:
from langchain_openai import ChatOpenAI
from langchain_neo4j import GraphCypherQAChain, Neo4jGraph

# LangChain 도구 활용 - LLM 및 그래프 객체 초기화
llm = ChatOpenAI(model="gpt-4.1-mini", temperature=0.0)

graph = Neo4jGraph(
    url=os.getenv("NEO4J_URI"),
    username=os.getenv("NEO4J_USERNAME"),
    password=os.getenv("NEO4J_PASSWORD"),
    database=os.getenv("NEO4J_DATABASE"),
    enhanced_schema=True,
)

# LangChain 도구 활용 - GraphCypherQAChain 객체 초기화
chain = GraphCypherQAChain.from_llm(
    llm=llm, 
    graph=graph, 
    allow_dangerous_requests=True,  # ⚠️ 학습 목적으로만 사용. 프로덕션에서는 쿼리 검증 필요
    verbose=True,)

result = chain.invoke({"query": "케이비자산운용은 모두 몇개의 ETF를 운용하고 있나요?"})



> Entering new GraphCypherQAChain chain...
Generated Cypher:
MATCH (c:Company {name: "케이비자산운용"})-[:MANAGES]->(e:ETF)
RETURN count(e) AS etfCount
Full Context:
[{'etfCount': 118}]

> Finished chain.


In [26]:
# 결과 출력
print(result)

{'query': '케이비자산운용은 모두 몇개의 ETF를 운용하고 있나요?', 'result': '케이비자산운용은 총 118개의 ETF를 운용하고 있습니다.'}


케이비, 운용, ETF

In [27]:
chain.invoke({"query": "케이비운용은 모두 몇개의 ETF를 운용하고 있나요?"})



> Entering new GraphCypherQAChain chain...
Generated Cypher:
MATCH (c:Company {name: "케이비운용"})-[:MANAGES]->(e:ETF)
RETURN count(e) AS etfCount
Full Context:
[{'etfCount': 0}]

> Finished chain.


{'query': '케이비운용은 모두 몇개의 ETF를 운용하고 있나요?',
 'result': '케이비운용은 총 0개의 ETF를 운용하고 있습니다.'}

In [28]:
result = chain.invoke("KOSPI 200을 기초 지수로 하는 ETF는 무엇인가요?")



> Entering new GraphCypherQAChain chain...
Generated Cypher:
MATCH (e:ETF) WHERE e.baseIndex CONTAINS "KOSPI 200" RETURN e
Full Context:
[]

> Finished chain.


In [29]:
print(graph.schema)

Node properties:
- **ETF**
  - `code`: STRING Example: "0000D0"
  - `name`: STRING Example: "TIGER 엔비디아미국채커버드콜밸런스(합성)"
  - `baseIndex`: STRING Example: "KEDI 엔비디아미국채30년타겟커버드콜혼합지수(TR)"
  - `category`: STRING Example: "혼합자산-주식+채권"
  - `company`: STRING Example: "미래에셋자산운용"
  - `disparityRatio`: FLOAT Min: -2.87, Max: 18010.07
  - `listingDate`: STRING Example: "2024/12/17"
  - `netAsset`: INTEGER Min: 89080687, Max: 9591343573385
  - `replicationMethod`: STRING Available options: ['합성(패시브)', '실물(패시브)', '실물(액티브)', '합성(액티브)']
  - `taxType`: STRING Available options: ['배당소득세(보유기간과세)', '비과세', '배당소득세(해외주식투자전용ETF)', '배당소득세(분리과세부동산ETF)', '비과세(분리과세부동산ETF)']
  - `totalFee`: FLOAT Min: 0.0, Max: 0.99
  - `trackingError`: FLOAT Min: 0.0, Max: 44.42
  - `volatility`: STRING Available options: ['매우낮음', '높음', '매우높음', '낮음', '보통']
  - `yearReturn`: FLOAT Min: -75.66, Max: 207.87
- **Company**
  - `name`: STRING Example: "미래에셋자산운용"
- **Category**
  - `name`: STRING Example: "혼합자산-주식+채권"
Relationship prope

In [30]:
# 결과 출력
print(result)

{'query': 'KOSPI 200을 기초 지수로 하는 ETF는 무엇인가요?', 'result': 'KOSPI 200을 기초 지수로 하는 ETF는 KODEX 200, TIGER 200, KBSTAR 200 등이 있습니다.'}


In [31]:
print(result['result'])

KOSPI 200을 기초 지수로 하는 ETF는 KODEX 200, TIGER 200, KBSTAR 200 등이 있습니다.


## 4. 인덱스 생성
- 자주 쿼리하는 속성에 대해 인덱스를 생성하여 쿼리 성능 개선

> 🔑 **Neo4j 인덱스 3종 한눈에 보기**
>
> | 종류 | 용도 | 문법 예 |
> |------|------|----------|
> | **Range (B-tree)** | `=`, `<`, `>`, `ORDER BY` | `CREATE INDEX FOR (n:ETF) ON (n.returnRate)` |
> | **Text (CONTAINS/STARTS WITH)** | 접두사·부분 문자열 | `CREATE TEXT INDEX FOR (n:ETF) ON (n.name)` |
> | **Full-text** | 형태소 검색·한글(`cjk`) | `CREATE FULLTEXT INDEX ... WITH {analyzer:'cjk'}` |
> | **Vector** | 임베딩 유사도 | `CREATE VECTOR INDEX ... dimensions:1536` |
>
> 👉 **용도별 선택 기준**: 정확 매칭/범위 → Range, 한글 형태소 검색 → Full-text(cjk), 의미 유사 → Vector.


`(1) 기본 인덱스`
- 유일성 제약조건이 있는 속성은 자동으로 인덱스가 생성되므로 별도 생성 불필요


In [32]:
# 인덱스 확인
graph.query("SHOW INDEXES")

[{'id': 0,
  'name': 'category_name_unique',
  'state': 'ONLINE',
  'populationPercent': 100.0,
  'type': 'RANGE',
  'entityType': 'NODE',
  'labelsOrTypes': ['Category'],
  'properties': ['name'],
  'indexProvider': 'range-1.0',
  'owningConstraint': 'category_name_unique',
  'lastRead': neo4j.time.DateTime(2026, 4, 30, 12, 20, 10, 373000000, tzinfo=<UTC>),
  'readCount': 1084},
 {'id': 2,
  'name': 'etf_code_unique',
  'state': 'ONLINE',
  'populationPercent': 100.0,
  'type': 'RANGE',
  'entityType': 'NODE',
  'labelsOrTypes': ['ETF'],
  'properties': ['code'],
  'indexProvider': 'range-1.0',
  'owningConstraint': 'etf_code_unique',
  'lastRead': neo4j.time.DateTime(2026, 4, 30, 12, 20, 14, 269000000, tzinfo=<UTC>),
  'readCount': 2796}]

`(2) 인덱스 활용한 검색`

- 인덱스를 생성한 속성을 WHERE 절에서 사용하면 Neo4j가 자동으로 인덱스를 활용

In [33]:
# 종목코드로 ETF 검색 (정확한 일치 검색)
code_search_query = """
MATCH (e:ETF)
WHERE e.code = '451060'
RETURN e AS etf
"""
graph.query(code_search_query)

[{'etf': {'trackingError': 0.77,
   'netAsset': 99754348820,
   'replicationMethod': '실물(액티브)',
   'listingDate': '2023/01/31',
   'code': '451060',
   'baseIndex': '코스피 200',
   'volatility': '높음',
   'yearReturn': -3.66,
   'disparityRatio': -0.01,
   'totalFee': 0.18,
   'name': '1Q K200액티브',
   'company': '하나자산운용',
   'category': '주식-시장대표',
   'taxType': '배당소득세(보유기간과세)'}}]

In [34]:
# 종목코드로 ETF 검색 (범위 검색)
code_search_query = """
MATCH (e:ETF)
WHERE toFloat(e.code) >= 453000 AND toFloat(e.code) < 454000   // 코드가 문자열 순으로 '453'와 '454' 사이에 있는 ETF 검색
RETURN e.code AS etf_code, e.name AS etf_name  // 종목코드와 종목명 반환     
ORDER BY e.code ASC       // 종목코드 기준으로 정렬 (ASC/DESC)
LIMIT 10     // 최대 10개 결과 반환
"""
results = graph.query(code_search_query)

results = pd.DataFrame(results)
results

,etf_code,etf_name
0,453010,PLUS KOFR금리
1,453060,HANARO KOFR금리액티브(합성)
2,453080,KOSEF 미국나스닥100(H)
3,453330,RISE 미국S&P500(H)
4,453540,TIGER 25-10 회사채(A+이상)액티브
5,453630,KODEX 미국S&P500필수소비재
6,453640,KODEX 미국S&P500헬스케어
7,453650,KODEX 미국S&P500금융
8,453660,KODEX 미국S&P500경기소비재
9,453810,KODEX 인도Nifty50


In [35]:
print(graph.schema)

Node properties:
- **ETF**
  - `code`: STRING Example: "0000D0"
  - `name`: STRING Example: "TIGER 엔비디아미국채커버드콜밸런스(합성)"
  - `baseIndex`: STRING Example: "KEDI 엔비디아미국채30년타겟커버드콜혼합지수(TR)"
  - `category`: STRING Example: "혼합자산-주식+채권"
  - `company`: STRING Example: "미래에셋자산운용"
  - `disparityRatio`: FLOAT Min: -2.87, Max: 18010.07
  - `listingDate`: STRING Example: "2024/12/17"
  - `netAsset`: INTEGER Min: 89080687, Max: 9591343573385
  - `replicationMethod`: STRING Available options: ['합성(패시브)', '실물(패시브)', '실물(액티브)', '합성(액티브)']
  - `taxType`: STRING Available options: ['배당소득세(보유기간과세)', '비과세', '배당소득세(해외주식투자전용ETF)', '배당소득세(분리과세부동산ETF)', '비과세(분리과세부동산ETF)']
  - `totalFee`: FLOAT Min: 0.0, Max: 0.99
  - `trackingError`: FLOAT Min: 0.0, Max: 44.42
  - `volatility`: STRING Available options: ['매우낮음', '높음', '매우높음', '낮음', '보통']
  - `yearReturn`: FLOAT Min: -75.66, Max: 207.87
- **Company**
  - `name`: STRING Example: "미래에셋자산운용"
- **Category**
  - `name`: STRING Example: "혼합자산-주식+채권"
Relationship prope

In [36]:
# 종목명으로 ETF 검색 (접두사 검색)
name_search_query = """
MATCH (e:ETF)  
WHERE e.name STARTS WITH 'TIGER'     // 'TIGER'로 시작하는 종목명 검색
RETURN e.name AS etf_name, e.code AS etf_code
"""

graph.query(name_search_query)

[{'etf_name': 'TIGER 엔비디아미국채커버드콜밸런스(합성)', 'etf_code': '0000D0'},
 {'etf_name': 'TIGER 은행', 'etf_code': '091220'},
 {'etf_name': 'TIGER 반도체', 'etf_code': '091230'},
 {'etf_name': 'TIGER 방송통신', 'etf_code': '098560'},
 {'etf_name': 'TIGER 200', 'etf_code': '102110'},
 {'etf_name': 'TIGER 라틴35', 'etf_code': '105010'},
 {'etf_name': 'TIGER 국채3년', 'etf_code': '114820'},
 {'etf_name': 'TIGER 차이나항셍25', 'etf_code': '117690'},
 {'etf_name': 'TIGER 인버스', 'etf_code': '123310'},
 {'etf_name': 'TIGER 레버리지', 'etf_code': '123320'},
 {'etf_name': 'TIGER 원유선물Enhanced(H)', 'etf_code': '130680'},
 {'etf_name': 'TIGER 미국나스닥100', 'etf_code': '133690'},
 {'etf_name': 'TIGER 농산물선물Enhanced(H)', 'etf_code': '137610'},
 {'etf_name': 'TIGER 삼성그룹펀더멘털', 'etf_code': '138520'},
 {'etf_name': 'TIGER LG그룹+펀더멘털', 'etf_code': '138530'},
 {'etf_name': 'TIGER 현대차그룹+펀더멘털', 'etf_code': '138540'},
 {'etf_name': 'TIGER 200 건설', 'etf_code': '139220'},
 {'etf_name': 'TIGER 200 중공업', 'etf_code': '139230'},
 {'etf_name': 'TIGER 20

In [37]:
# 운용사 이름으로 ETF 검색 (IN 연산자 활용)
company_search_query = """
MATCH (e:ETF)  
WHERE e.company IN ['삼성자산운용', '미래에셋자산운용']  // 삼성자산운용 또는 미래에셋자산운용이 운용하는 ETF 검색" 
RETURN e.name, e.company  // 종목명과 운용사 이름 반환
LIMIT 10   // 최대 10개 결과 반환
"""

graph.query(company_search_query)

[{'e.name': 'TIGER 엔비디아미국채커버드콜밸런스(합성)', 'e.company': '미래에셋자산운용'},
 {'e.name': 'KODEX 200', 'e.company': '삼성자산운용'},
 {'e.name': 'KODEX 반도체', 'e.company': '삼성자산운용'},
 {'e.name': 'KODEX 은행', 'e.company': '삼성자산운용'},
 {'e.name': 'KODEX 자동차', 'e.company': '삼성자산운용'},
 {'e.name': 'TIGER 은행', 'e.company': '미래에셋자산운용'},
 {'e.name': 'TIGER 반도체', 'e.company': '미래에셋자산운용'},
 {'e.name': 'TIGER 방송통신', 'e.company': '미래에셋자산운용'},
 {'e.name': 'KODEX 차이나H', 'e.company': '삼성자산운용'},
 {'e.name': 'KODEX 일본TOPIX100', 'e.company': '삼성자산운용'}]

In [38]:
# 카테고리 노드와 관련된 관계 검색
relationship_query = """
MATCH (c:Category)      
WHERE c.name CONTAINS '주식-시장대표'   // 카테고리 이름이 '주식-시장대표'인 카테고리 노드 검색
MATCH (e:ETF)-[r:BELONGS_TO]->(c)   // ETF 노드와 카테고리 노드 간의 BELONGS_TO 관계 검색
RETURN c.name AS category_name,  // 카테고리 이름 반환
       collect(e.name) AS etf_names,  // 관련된 ETF 노드의 이름을 리스트로 반환
       count(e) AS etf_count  // 관련된 ETF 개수 반환
"""

graph.query(relationship_query)

[{'category_name': '주식-시장대표',
  'etf_names': ['1Q K200액티브',
   '1Q 차이나H(H)',
   '1Q 코리아밸류업',
   'ACE 200',
   'ACE 200TR',
   'ACE 러시아MSCI(합성)',
   'ACE 레버리지',
   'ACE 멕시코MSCI(합성)',
   'ACE 미국S&P500',
   'ACE 미국나스닥100',
   'ACE 베트남VN30(합성)',
   'ACE 아시아TOP50S&P',
   'ACE 인도네시아MSCI(합성)',
   'ACE 인버스',
   'ACE 일본Nikkei225(H)',
   'ACE 일본TOPIX레버리지(H)',
   'ACE 일본TOPIX인버스(합성 H)',
   'ACE 중국과창판STAR50',
   'ACE 중국본토CSI300',
   'ACE 중국본토CSI300레버리지(합성)',
   'ACE 코리아밸류업',
   'ACE 코스닥150',
   'ACE 코스피',
   'ACE 필리핀MSCI(합성)',
   'DAISHIN343 K200',
   'FOCUS AI코리아액티브',
   'FOCUS KRX300',
   'HANARO 200',
   'HANARO 200TR',
   'HANARO 200선물레버리지',
   'HANARO 200선물레버리지1.5X',
   'HANARO 200선물인버스',
   'HANARO KRX300',
   'HANARO MSCI Korea TR',
   'HANARO 미국S&P500',
   'HANARO 코리아밸류업',
   'HANARO 코스닥150',
   'HANARO 코스닥150선물레버리지',
   'HANARO 코스닥150선물레버리지1.5X',
   'HK 200',
   'HK 베스트일레븐액티브',
   'ITF 200',
   'KODEX 200',
   'KODEX 200TR',
   'KODEX 200동일가중',
   'KODEX 200선물인버스2X',
   'KODEX 200액티브',
  

`(3) 인덱스 생성`

- 대량의 데이터를 처리하거나 자주 쿼리하는 속성에 대해 인덱스를 생성하면 쿼리 성능이 크게 향상됨

In [39]:
# 수익률에 대한 인덱스 생성 - 수익률 기반 검색 및 정렬 시 성능 향상
return_index_query = """
CREATE INDEX etf_yearreturn_idx IF NOT EXISTS
FOR (e:ETF) ON (e.yearReturn)
"""
graph.query(return_index_query)

[]

In [40]:
# 수익률 기준으로 ETF 검색 (정렬)
return_search_query = """
MATCH (e:ETF)
WHERE e.yearReturn > 10.0   // 수익률이 10% 이상인 ETF 검색
RETURN e.name AS Name, e.yearReturn AS Return   // 종목명과 수익률 반환
ORDER BY e.yearReturn DESC   // 수익률 기준으로 내림차순 정렬
LIMIT 10   // 최대 10개 결과 반환
"""

graph.query(return_search_query)

[{'Name': 'ACE 미국빅테크TOP7 Plus레버리지(합성)', 'Return': 207.87},
 {'Name': 'PLUS 미국테크TOP10레버리지(합성)', 'Return': 179.94},
 {'Name': 'TIGER 미국나스닥100레버리지(합성)', 'Return': 106.09},
 {'Name': 'TIMEFOLIO 미국나스닥100액티브', 'Return': 97.75},
 {'Name': 'TIMEFOLIO 글로벌AI인공지능액티브', 'Return': 91.95},
 {'Name': 'HANARO 글로벌생성형AI액티브', 'Return': 88.45},
 {'Name': 'ACE 미국빅테크TOP7 Plus', 'Return': 85.06},
 {'Name': '에셋플러스 글로벌플랫폼액티브', 'Return': 84.22},
 {'Name': 'KODEX 미국메타버스나스닥액티브', 'Return': 80.51},
 {'Name': 'TIGER 글로벌AI액티브', 'Return': 77.62}]

In [41]:
# 순자산총액에 대한 인덱스 생성 - 규모별 검색 및 정렬 시 성능 향상
netasset_index_query = """
CREATE INDEX etf_netasset_idx IF NOT EXISTS
FOR (e:ETF) ON (e.netAsset)
"""
graph.query(netasset_index_query)

[]

In [42]:
# 순자산총액 기준으로 ETF 검색 (정렬)
netasset_search_query = """
MATCH (e:ETF)
WHERE e.netAsset > 1000000000   // 순자산총액이 10억 이상인 ETF 검색
RETURN e.name, e.netAsset   // 종목명과 순자산총액 반환
ORDER BY e.netAsset DESC   // 순자산총액 기준으로 내림차순 정렬
LIMIT 10   // 최대 10개 결과 반환
"""
graph.query(netasset_search_query)

[{'e.name': 'KODEX CD금리액티브(합성)', 'e.netAsset': 9591343573385},
 {'e.name': 'TIGER CD금리투자KIS(합성)', 'e.netAsset': 6871517745155},
 {'e.name': 'TIGER 미국S&P500', 'e.netAsset': 6433623514411},
 {'e.name': 'KODEX 200', 'e.netAsset': 5604796801979},
 {'e.name': 'TIGER 미국나스닥100', 'e.netAsset': 4494225408877},
 {'e.name': 'KODEX KOFR금리액티브(합성)', 'e.netAsset': 4331602537465},
 {'e.name': 'KODEX 머니마켓액티브', 'e.netAsset': 4004995260449},
 {'e.name': 'TIGER KOFR금리액티브(합성)', 'e.netAsset': 3477409947171},
 {'e.name': 'TIGER 미국테크TOP10 INDXX', 'e.netAsset': 3144375360249},
 {'e.name': 'KODEX 미국S&P500TR', 'e.netAsset': 3050770738959}]

`(4) 복합 인덱스`

- 여러 속성을 함께 검색하는 경우가 많다면 복합 인덱스를 고려

In [43]:
# 복합 인덱스 생성 - 카테고리와 수익률을 동시에 검색 및 정렬 시 성능 향상
compound_index_query = """
CREATE INDEX etf_category_return_idx IF NOT EXISTS
FOR (e:ETF) ON (e.category, e.yearReturn)
"""
graph.query(compound_index_query)

[]

In [44]:
# 카테고리와 수익률 기준으로 ETF 검색 (정렬)
compound_search_query = """
MATCH (e:ETF)
WHERE e.category = '주식-시장대표' AND e.yearReturn > 10.0   // 카테고리가 '주식-시장대표'이고 수익률이 10% 이상인 ETF 검색
RETURN e.name, e.yearReturn   // 종목명과 수익률 반환
ORDER BY e.yearReturn DESC   // 수익률 기준으로 내림차순 정렬
LIMIT 10   // 최대 10개 결과 반환
"""
graph.query(compound_search_query)

[{'e.name': 'TIGER 미국나스닥100레버리지(합성)', 'e.yearReturn': 106.09},
 {'e.name': 'TIMEFOLIO 미국나스닥100액티브', 'e.yearReturn': 97.75},
 {'e.name': '에셋플러스 글로벌플랫폼액티브', 'e.yearReturn': 84.22},
 {'e.name': 'TIMEFOLIO 미국S&P500액티브', 'e.yearReturn': 71.73},
 {'e.name': 'KODEX 미국나스닥100레버리지(합성 H)', 'e.yearReturn': 63.12},
 {'e.name': '에셋플러스 글로벌대장장이액티브', 'e.yearReturn': 59.98},
 {'e.name': 'TIGER 미국S&P500레버리지(합성 H)', 'e.yearReturn': 54.0},
 {'e.name': 'TIGER 차이나CSI300레버리지(합성)', 'e.yearReturn': 50.87},
 {'e.name': 'KODEX 미국나스닥100TR', 'e.yearReturn': 48.77},
 {'e.name': 'ACE 중국본토CSI300레버리지(합성)', 'e.yearReturn': 48.62}]

`(5) Full Text 검색 인덱스`

- 텍스트 검색이 필요한 경우  Full Text 인덱스를 고려

In [45]:
# Full Text 없이 종목명 검색 ("액티브" 포함)
name_search_query = """
MATCH (e:ETF)  
WHERE e.name CONTAINS '액티브'   // 종목명에 '액티브'가 포함된 ETF 검색
RETURN e.name   // 종목명 반환
LIMIT 10   // 최대 10개 결과 반환
"""
graph.query(name_search_query)

[{'e.name': 'RISE 단기국공채액티브'},
 {'e.name': 'RISE 중장기국공채액티브'},
 {'e.name': 'TIGER 단기채권액티브'},
 {'e.name': 'ACE 중장기국공채액티브'},
 {'e.name': 'KODEX 종합채권(AA-이상)액티브'},
 {'e.name': 'KODEX 단기변동금리부채권액티브'},
 {'e.name': 'PLUS 단기채권액티브'},
 {'e.name': 'TIGER 미국달러단기채권액티브'},
 {'e.name': 'RISE 금융채액티브'},
 {'e.name': 'ACE 종합채권(AA-이상)KIS액티브'}]

In [46]:
# Full Text 인덱스 생성 - 종목명에 대한 Full Text 검색을 위한 인덱스 생성
fulltext_index_query = """
CREATE FULLTEXT INDEX etf_name_fulltext IF NOT EXISTS
FOR (e:ETF) ON EACH [e.name]
"""
graph.query(fulltext_index_query)

[]

In [47]:
graph.query("SHOW INDEXES")

[{'id': 0,
  'name': 'category_name_unique',
  'state': 'ONLINE',
  'populationPercent': 100.0,
  'type': 'RANGE',
  'entityType': 'NODE',
  'labelsOrTypes': ['Category'],
  'properties': ['name'],
  'indexProvider': 'range-1.0',
  'owningConstraint': 'category_name_unique',
  'lastRead': neo4j.time.DateTime(2026, 4, 30, 12, 20, 10, 373000000, tzinfo=<UTC>),
  'readCount': 1084},
 {'id': 8,
  'name': 'etf_category_return_idx',
  'state': 'ONLINE',
  'populationPercent': 100.0,
  'type': 'RANGE',
  'entityType': 'NODE',
  'labelsOrTypes': ['ETF'],
  'properties': ['category', 'yearReturn'],
  'indexProvider': 'range-1.0',
  'owningConstraint': None,
  'lastRead': None,
  'readCount': None},
 {'id': 2,
  'name': 'etf_code_unique',
  'state': 'ONLINE',
  'populationPercent': 100.0,
  'type': 'RANGE',
  'entityType': 'NODE',
  'labelsOrTypes': ['ETF'],
  'properties': ['code'],
  'indexProvider': 'range-1.0',
  'owningConstraint': 'etf_code_unique',
  'lastRead': neo4j.time.DateTime(2026, 

In [48]:
# Full Text 검색 - 종목명에 "액티브" 포함
fulltext_search_query = """
CALL db.index.fulltext.queryNodes('etf_name_fulltext', '펀드 액티브') YIELD node  // Full Text 인덱스 검색
RETURN node.name   // 종목명 반환
LIMIT 10   // 최대 10개 결과 반환
"""
graph.query(fulltext_search_query)

[{'node.name': 'KODEX 1년은행양도성예금증서+액티브(합성)'},
 {'node.name': 'UNICORN R&D 액티브'},
 {'node.name': 'HANARO 종합채권(AA-이상)액티브'},
 {'node.name': 'RISE 단기종합채권(AA-이상)액티브'},
 {'node.name': '히어로즈 종합채권(AA-이상)액티브'},
 {'node.name': '파워 종합채권(AA-이상)액티브'},
 {'node.name': 'RISE 종합채권(A-이상)액티브'},
 {'node.name': 'SOL 종합채권(AA-이상)액티브'},
 {'node.name': 'HK 종합채권(AA-이상)액티브'},
 {'node.name': 'KODEX ESG종합채권(A-이상)액티브'}]

In [49]:
graph.query("SHOW FULLTEXT INDEXES")  # 생성된 Full Text 인덱스 확인

[{'id': 9,
  'name': 'etf_name_fulltext',
  'state': 'ONLINE',
  'populationPercent': 100.0,
  'type': 'FULLTEXT',
  'entityType': 'NODE',
  'labelsOrTypes': ['ETF'],
  'properties': ['name'],
  'indexProvider': 'fulltext-1.0',
  'owningConstraint': None,
  'lastRead': None,
  'readCount': None}]

In [50]:
# 인덱스 삭제
drop_index_query = """
DROP INDEX etf_name_baseindex_fulltext IF EXISTS
"""
graph.query(drop_index_query)

[]

In [51]:
# Full Text 인덱스 생성 - 종목명, 기초지수에 대한 Full Text 검색을 위한 인덱스 생성 (cjk 형태소 분석기 사용)
fulltext_index_query = """
CREATE FULLTEXT INDEX etf_name_baseindex_fulltext IF NOT EXISTS
FOR (e:ETF) ON EACH [e.name, e.baseIndex]
OPTIONS {
    indexConfig: {
        `fulltext.analyzer`: 'cjk',   // cjk 형태소 분석기 사용 
        `fulltext.eventually_consistent`: true  // 성능 최적화를 위한 설정 활성화
    }
}
"""
graph.query(fulltext_index_query)

[]

In [52]:
graph.query("SHOW FULLTEXT INDEXES")  # 생성된 Full Text 인덱스 확인

[{'id': 10,
  'name': 'etf_name_baseindex_fulltext',
  'state': 'POPULATING',
  'populationPercent': 0.0,
  'type': 'FULLTEXT',
  'entityType': 'NODE',
  'labelsOrTypes': ['ETF'],
  'properties': ['name', 'baseIndex'],
  'indexProvider': 'fulltext-1.0',
  'owningConstraint': None,
  'lastRead': None,
  'readCount': None},
 {'id': 9,
  'name': 'etf_name_fulltext',
  'state': 'ONLINE',
  'populationPercent': 100.0,
  'type': 'FULLTEXT',
  'entityType': 'NODE',
  'labelsOrTypes': ['ETF'],
  'properties': ['name'],
  'indexProvider': 'fulltext-1.0',
  'owningConstraint': None,
  'lastRead': None,
  'readCount': None}]

In [53]:
# Full Text 검색 - 종목명 또는 기초지수에 "클린 에너지" 포함
fulltext_search_query = """
CALL db.index.fulltext.queryNodes('etf_name_baseindex_fulltext', '펀드액티브') YIELD node  // Full Text 인덱스 검색
RETURN node.name, node.baseIndex   // 종목명과 기초지수 반환
LIMIT 10   // 최대 10개 결과 반환
"""
graph.query(fulltext_search_query)

[{'node.name': 'ACE 미국하이일드액티브(H)',
  'node.baseIndex': 'Markit iBoxx USD Liquid High Yield Total Return Index'},
 {'node.name': 'TIMEFOLIO 글로벌소비트렌드액티브',
  'node.baseIndex': 'Solactive 뉴에이지컨수머지수'},
 {'node.name': 'KODEX 1년은행양도성예금증서+액티브(합성)',
  'node.baseIndex': 'KAP 1년은행 CD+추가금리 지수(총수익지수)'},
 {'node.name': 'KODEX ESG종합채권(A-이상)액티브',
  'node.baseIndex': 'KAP ESG 종합채권지수(A- 이상, 총수익)'},
 {'node.name': 'UNICORN R&D 액티브', 'node.baseIndex': '코스피 200'},
 {'node.name': 'ACE 5월만기자동연장회사채AA-이상액티브',
  'node.baseIndex': 'KIS 5월 만기자동연장회사채(AA-이상) 총수익지수'},
 {'node.name': 'ACE 8월만기자동연장회사채AA-이상액티브',
  'node.baseIndex': 'KAP 8월 만기자동연장회사채(AA-이상) 총수익지수'},
 {'node.name': 'RISE 금융채액티브',
  'node.baseIndex': 'KIS 종합채권금융채 2.5Y-3Y 지수 (총수익)'},
 {'node.name': 'ACE 11월만기자동연장회사채AA-이상액티브',
  'node.baseIndex': 'KIS 11월 만기자동연장회사채(AA-이상) 총수익지수'},
 {'node.name': 'ACE 2월만기자동연장회사채AA-이상액티브',
  'node.baseIndex': 'KAP 2월 만기자동연장회사채(AA-이상) 총수익지수'}]

In [54]:
# cjk 애널라이저 사용해서, etf.name, etf.company 속성에 대한 Full-text Index를 만들고 테스트 (~ 20:40분까지)

fulltext_index_query = """
CREATE FULLTEXT INDEX etf_name_company_fulltext IF NOT EXISTS
FOR (e:ETF) ON EACH [e.name, e.company]
OPTIONS {
    indexConfig: {
        `fulltext.analyzer`: 'cjk',   // cjk 형태소 분석기 사용 
        `fulltext.eventually_consistent`: true  // 성능 최적화를 위한 설정 활성화
    }
}
"""
graph.query(fulltext_index_query)


[]

In [55]:
# Full Text 검색 
fulltext_search_query = """
CALL db.index.fulltext.queryNodes('etf_name_company_fulltext', '채권') YIELD node  // Full Text 인덱스 검색
RETURN node.name, node.company
LIMIT 10   // 최대 10개 결과 반환
"""
graph.query(fulltext_search_query)

[{'node.name': 'KODEX 단기채권', 'node.company': '삼성자산운용'},
 {'node.name': 'RISE 채권혼합', 'node.company': '케이비자산운용'},
 {'node.name': 'PLUS 애플채권혼합', 'node.company': '한화자산운용'},
 {'node.name': 'ACE 종합채권(AA-이상)KIS액티브', 'node.company': '한국투자신탁운용'},
 {'node.name': 'PLUS 단기채권액티브', 'node.company': '한화자산운용'},
 {'node.name': 'RISE 종합채권(A-이상)액티브', 'node.company': '케이비자산운용'},
 {'node.name': 'HANARO 단기채권액티브', 'node.company': '엔에이치아문디자산운용'},
 {'node.name': 'TIGER 단기채권액티브', 'node.company': '미래에셋자산운용'},
 {'node.name': 'TIGER 경기방어채권혼합', 'node.company': '미래에셋자산운용'},
 {'node.name': 'SOL 초단기채권액티브', 'node.company': '신한자산운용'}]

`(6) 벡터 인덱스 검색`

- 벡터 검색이 필요한 경우 벡터 인덱스를 고려
- 단일 속성에 대해서만 벡터 인덱스를 지원

In [56]:
from langchain_core.documents import Document

etfs = graph.query("""
MATCH (e:ETF)   // 모든 ETF 노드 검색
RETURN e.name AS name,   // 종목명 반환
       e.company AS company,   // 운용사 반환
       e.baseIndex AS baseIndex,   // 기초지수 반환
       e.category AS category,   // 카테고리 반환
       e.listingDate AS listingDate   // 상장일 반환
""")

# 검색 결과를 DataFrame으로 변환
etfs_df = pd.DataFrame(etfs)

# 종목명, 운용사, 기초지수 속성을 결합한 문서를 생성
docs = [
    Document(
        page_content=f"종목명: {row['name']}, 운용사: {row['company']}, 기초지수: {row['baseIndex']}",
        metadata={"name": row['name'], "company": row['company'], "baseIndex": row['baseIndex'], "category": row['category'], "listingDate": row['listingDate']}
    )
    for _, row in etfs_df.iterrows()
]

# 문서의 속성 확인
for doc in docs[:5]:  # 상위 5개 문서 출력
    print(doc.page_content)
    print(doc.metadata)
    print()

종목명: TIGER 엔비디아미국채커버드콜밸런스(합성), 운용사: 미래에셋자산운용, 기초지수: KEDI 엔비디아미국채30년타겟커버드콜혼합지수(TR)
{'name': 'TIGER 엔비디아미국채커버드콜밸런스(합성)', 'company': '미래에셋자산운용', 'baseIndex': 'KEDI 엔비디아미국채30년타겟커버드콜혼합지수(TR)', 'category': '혼합자산-주식+채권', 'listingDate': '2024/12/17'}

종목명: KODEX 200, 운용사: 삼성자산운용, 기초지수: 코스피 200
{'name': 'KODEX 200', 'company': '삼성자산운용', 'baseIndex': '코스피 200', 'category': '주식-시장대표', 'listingDate': '2002/10/14'}

종목명: KOSEF 200, 운용사: 키움투자자산운용, 기초지수: 코스피 200
{'name': 'KOSEF 200', 'company': '키움투자자산운용', 'baseIndex': '코스피 200', 'category': '주식-시장대표', 'listingDate': '2002/10/14'}

종목명: KODEX 반도체, 운용사: 삼성자산운용, 기초지수: KRX 반도체
{'name': 'KODEX 반도체', 'company': '삼성자산운용', 'baseIndex': 'KRX 반도체', 'category': '주식-업종섹터-정보기술', 'listingDate': '2006/06/27'}

종목명: KODEX 은행, 운용사: 삼성자산운용, 기초지수: KRX 은행
{'name': 'KODEX 은행', 'company': '삼성자산운용', 'baseIndex': 'KRX 은행', 'category': '주식-업종섹터-금융', 'listingDate': '2006/06/27'}



In [57]:
from langchain_openai import OpenAIEmbeddings

# OpenAI 임베딩 모델 초기화
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

# 기존 ETF 데이터 조회
etfs = graph.query("""
MATCH (e:ETF)
RETURN e.name AS name,
       e.company AS company,
       e.baseIndex AS baseIndex,
       e.category AS category,
       e.listingDate AS listingDate
""")

# 각 ETF에 대해 임베딩 생성 및 업데이트
for etf in tqdm(etfs):
    # ETF 정보를 텍스트로 결합
    combined_text = f"이 종목은 {etf['name']}이며, 운용사는 {etf['company']}입니다. 기초지수는 {etf['baseIndex']}입니다."
    
    # 임베딩 생성
    embedding_vector = embeddings.embed_query(combined_text)
    
    # 기존 ETF 노드에 임베딩 속성 추가
    graph.query("""
    MATCH (e:ETF {name: $name})
    CALL db.create.setNodeVectorProperty(e, 'embedding', $embedding)
    """, params={
        'name': etf['name'],
        'embedding': embedding_vector
    })

100%|██████████| 930/930 [03:41<00:00,  4.19it/s]


In [ ]:
# 각 ETF에 대해 임베딩 생성 및 업데이트
for etf in tqdm(etfs):
    try:
        # ETF 정보를 텍스트로 결합
        combined_text = f"이 종목은 {etf['name']}이며, 운용사는 {etf['company']}입니다. 기초지수는 {etf['baseIndex']}입니다."
        
        # 임베딩 생성
        embedding_vector = embeddings.embed_query(combined_text)
        
        # 기존 ETF 노드에 임베딩 속성 추가 (Case 2: combined_text를 별도 속성으로 추가)
        graph.query("""
        MATCH (e:ETF {name: $name})
        SET e.embedding_text = $embedding_text
        CALL db.create.setNodeVectorProperty(e, 'embedding', $embedding)
        """, params={
            'name': etf['name'],
            'embedding': embedding_vector,
            "embedding_text": combined_text
        })
    except Exception as e:
        print(f"Error processing ETF {etf.get('name', 'Unknown')}: {e}")

In [59]:
# 각 ETF에 대해 임베딩 생성 및 업데이트
for etf in tqdm(etfs):
    try:
        # ETF 정보를 텍스트로 결합
        combined_text = f"이 종목은 {etf['name']}이며, 운용사는 {etf['company']}입니다. 기초지수는 {etf['baseIndex']}입니다."
        
        # 임베딩 생성
        embedding_vector = embeddings.embed_query(combined_text)
        
        # 기존 ETF 노드에 임베딩 속성 추가 (Case 2: combined_text를 별도 속성으로 추가)
        graph.query("""
        MATCH (e:ETF {name: $name})
        SET e.embedding_text = $embedding_text
        WITH e
        CALL db.create.setNodeVectorProperty(e, 'embedding', $embedding)
        """, params={
            'name': etf['name'],
            'embedding': embedding_vector,
            "embedding_text": combined_text
        })
    except Exception as e:
        print(f"Error processing ETF {etf.get('name', 'Unknown')}: {e}")

100%|██████████| 930/930 [03:22<00:00,  4.60it/s]


In [60]:
# 벡터 인덱스 생성
graph.query("""
CREATE VECTOR INDEX etf_vector_index IF NOT EXISTS
FOR (e:ETF) ON (e.embedding)
OPTIONS {
  indexConfig: {
    `vector.dimensions`: 1536,
    `vector.similarity_function`: 'cosine'
  }
}
""")

[]

In [61]:
# 기존 벡터 인덱스에 연결
from langchain_openai import OpenAIEmbeddings
from langchain_neo4j import Neo4jVector

embeddings = OpenAIEmbeddings(model="text-embedding-3-small") 

vector_db = Neo4jVector.from_existing_index(
    embeddings,
    url=NEO4J_URI,
    username=NEO4J_USERNAME,
    password=NEO4J_PASSWORD,
    database=NEO4J_DATABASE,   # etf 데이터베이스
    index_name="etf_vector_index",  # 위에서 생성한 인덱스 이름
    # node_label="ETF",  # 노드 레이블
    text_node_property="name",  # 텍스트 속성 (LangChain 문서의 page_content로 지정될 텍스트 속성)
    embedding_node_property="embedding"  # 임베딩 속성
)

# 유사도 검색 테스트
results = vector_db.similarity_search("KODEX", k=5)
for result in results:
    print(f"ETF: {result.page_content}")
    print(f"메타데이터: {result.metadata}")
    print("---")

ETF: KODEX IT
메타데이터: {'yearReturn': -17.87, 'listingDate': '2017/03/28', 'baseIndex': 'KRX 정보기술', 'code': '266370', 'disparityRatio': 0.05, 'volatility': '매우높음', 'category': '주식-업종섹터-정보기술', 'netAsset': 18249222663, 'totalFee': 0.45, 'company': '삼성자산운용', 'trackingError': 0.52, 'replicationMethod': '실물(패시브)', 'taxType': '비과세', 'embedding_text': '이 종목은 KODEX IT이며, 운용사는 삼성자산운용입니다. 기초지수는 KRX 정보기술입니다.'}
---
ETF: KODEX 미디어&엔터테인먼트
메타데이터: {'yearReturn': -7.1, 'listingDate': '2017/03/28', 'baseIndex': 'KRX 미디어&엔터테인먼트', 'code': '266360', 'disparityRatio': 0.04, 'volatility': '높음', 'category': '주식-업종섹터-정보기술', 'netAsset': 41406746008, 'totalFee': 0.45, 'company': '삼성자산운용', 'trackingError': 0.2, 'replicationMethod': '실물(패시브)', 'taxType': '비과세', 'embedding_text': '이 종목은 KODEX 미디어&엔터테인먼트이며, 운용사는 삼성자산운용입니다. 기초지수는 KRX 미디어&엔터테인먼트입니다.'}
---
ETF: KODEX 운송
메타데이터: {'yearReturn': 7.31, 'listingDate': '2011/04/26', 'baseIndex': 'KRX 운송', 'code': '140710', 'disparityRatio': -0.32, 'volatility': '높음', 'category'

In [62]:
# from langchain_openai import OpenAIEmbeddings
# from langchain_neo4j import Neo4jVector

# embeddings = OpenAIEmbeddings(model="text-embedding-3-small") 

# vector_db = Neo4jVector.from_documents(
#     docs,
#     embeddings,
#     url=NEO4J_URI,
#     username=NEO4J_USERNAME,
#     password=NEO4J_PASSWORD,
#     index_name="etf_name_company_asset_vector_index" # 인덱스 이름 설정 (선택 사항)
# )

In [63]:
query = "삼성자산운용의 국내 주식형 상품은 무엇인가요?" # 검색할 쿼리
similar_docs = vector_db.similarity_search_with_score(query, k=5) # 유사도 상위 5개 문서 검색

print(f"'{query}'와 유사한 문서:")
for doc, score in similar_docs:
    print(f"문서 내용: {doc.page_content}, 유사도 점수: {score}")
    print(f"메타데이터: {doc.metadata}")
    print("-" * 50)  # 구분선 출력

'삼성자산운용의 국내 주식형 상품은 무엇인가요?'와 유사한 문서:
문서 내용: KODEX MSCI밸류, 유사도 점수: 0.7427773475646973
메타데이터: {'yearReturn': -9.31, 'listingDate': '2017/07/11', 'baseIndex': 'MSCI Korea IMI Enhanced Value Capped', 'code': '275290', 'disparityRatio': -0.18, 'volatility': '보통', 'category': '주식-전략-가치', 'netAsset': 10998273192, 'totalFee': 0.26, 'company': '삼성자산운용', 'trackingError': 1.69, 'replicationMethod': '실물(패시브)', 'taxType': '비과세', 'embedding_text': '이 종목은 KODEX MSCI밸류이며, 운용사는 삼성자산운용입니다. 기초지수는 MSCI Korea IMI Enhanced Value Capped입니다.'}
--------------------------------------------------
문서 내용: KODEX MSCI Korea, 유사도 점수: 0.7418210506439209
메타데이터: {'yearReturn': -7.58, 'listingDate': '2012/04/30', 'baseIndex': 'MSCI Korea Index', 'code': '156080', 'disparityRatio': 0.01, 'volatility': '높음', 'category': '주식-시장대표', 'netAsset': 10544303239, 'totalFee': 0.09, 'company': '삼성자산운용', 'trackingError': 0.72, 'replicationMethod': '실물(패시브)', 'taxType': '비과세', 'embedding_text': '이 종목은 KODEX MSCI Korea이며, 운용사는 삼성자산운용입니

In [64]:
# 유사한 문서들의 메타데이터에서 카테고리와 운용사 정보 추출
categories = set()
companies = set()

for doc, score in similar_docs:
    # 벡터 검색된 문서의 메타데이터에서 정보 추출
    if 'category' in doc.metadata:
        categories.add(doc.metadata['category'])
    if 'company' in doc.metadata:
        companies.add(doc.metadata['company'])


categories

{'주식-시장대표', '주식-전략-가치', '혼합자산'}

In [65]:
companies

{'삼성자산운용'}

In [66]:
# 유사한 문서들의 메타데이터에서 카테고리와 운용사 정보 추출
categories = set()
companies = set()

for doc, score in similar_docs:
    # 벡터 검색된 문서의 메타데이터에서 정보 추출
    if 'category' in doc.metadata:
        categories.add(doc.metadata['category'])
    if 'company' in doc.metadata:
        companies.add(doc.metadata['company'])

# 쿼리에서 언급된 조건들도 직접 추가
target_company = "삼성자산운용"
target_categories = ["통화-미국달러", "채권-혼합-단기"]  # 가능한 카테고리 표현들

# ETF 노드에서 조건에 맞는 상품 검색 (특정 운용사가 관리하는 ETF 중 카테고리가 일치하는 상품)
etf_query = """
MATCH (e:ETF)
WHERE e.company = $target_company  // 자산운용사 매칭
  AND e.category IN $target_categories  // 카테고리 매칭
WITH e
ORDER BY toFloat(e.netAsset) DESC
LIMIT 10
RETURN 
    e.name AS name,
    e.company AS company,
    e.category AS category,
    e.baseIndex AS baseIndex,
    e.yearReturn AS yearReturn,
    e.totalFee AS totalFee,
    toFloat(e.netAsset) AS netAsset
"""

params = {
    "target_company": target_company,
    "target_categories": target_categories,
}

etf_results = graph.query(etf_query, params=params)
etf_df = pd.DataFrame(etf_results)
print(f"검색된 ETF 상품: {len(etf_df)}개")
etf_df.head(10)

검색된 ETF 상품: 7개


,name,company,category,baseIndex,yearReturn,totalFee,netAsset
0,KODEX 머니마켓액티브,삼성자산운용,채권-혼합-단기,KAP MMF 지수(TR),0.00,0.05,4.004995e+12
1,KODEX 단기채권PLUS,삼성자산운용,채권-혼합-단기,KRW Cash PLUS 지수(총수익),3.71,0.15,1.641191e+12
2,KODEX 단기변동금리부채권액티브,삼성자산운용,채권-혼합-단기,KAP 단기변동금리부은행채권지수,3.50,0.15,1.788297e+11
3,KODEX 미국달러선물인버스2X,삼성자산운용,통화-미국달러,미국달러선물지수,-21.48,0.41,1.551231e+11
4,KODEX 미국달러선물,삼성자산운용,통화-미국달러,미국달러선물지수,16.66,0.21,5.999443e+10
5,KODEX 미국달러선물인버스,삼성자산운용,통화-미국달러,미국달러선물지수,-9.85,0.41,4.743744e+10
6,KODEX 미국달러선물레버리지,삼성자산운용,통화-미국달러,미국달러선물지수,30.25,0.45,3.894008e+10


In [67]:
# ETF 전용 벡터 검색 설정
from langchain_openai import OpenAIEmbeddings
from langchain_neo4j import Neo4jVector

embeddings = OpenAIEmbeddings(model="text-embedding-3-small") 

vector_db = Neo4jVector.from_existing_index(
    embeddings,
    url=NEO4J_URI,
    username=NEO4J_USERNAME,
    password=NEO4J_PASSWORD,
    database=NEO4J_DATABASE,
    index_name="etf_vector_index",  # ETF 벡터 인덱스
    node_label="ETF",  # ETF 노드 레이블
    text_node_property="name",  # 기본 텍스트 속성 (ETF 이름)
    embedding_node_property="embedding",  # 임베딩 속성
    retrieval_query="""
    // 기본 ETF 노드와 점수
    WITH node, score
    
    // ETF와 관련된 모든 정보 수집
    OPTIONAL MATCH (node)-[:BELONGS_TO]->(category:Category)
    OPTIONAL MATCH (node)<-[:MANAGES]-(company:Company)
    
    // 관련 ETF들도 찾기 (같은 카테고리에 속하는)
    OPTIONAL MATCH (node)-[:BELONGS_TO]->(cat:Category)<-[:BELONGS_TO]-(related_etf:ETF)
    WHERE related_etf <> node       // 같은 ETF 제외
    
    // 집계 함수를 사용하여 관련 ETF들 수집
    WITH node, score,
         collect(DISTINCT related_etf.name)[0..3] AS related_etfs   // 관련 ETF 3개만 (중복 제외)
    
    // 검색용 종합 텍스트 생성
    WITH node, score, related_etfs,
         "ETF 종목: " + node.name + "\n" + 
         "- 운용사: " + COALESCE(node.company, "N/A") + "\n" +
         "- 기초지수: " + COALESCE(node.baseIndex, "N/A") + "\n" +
         "- 카테고리: " + COALESCE(node.category, "N/A") + "\n" +
         "- 관련 ETF:" + apoc.text.join(related_etfs, ", ") + "\n" AS combined_text

    RETURN combined_text AS text,   // 검색 결과 텍스트 (Langchain Document의 page_content 속성)
           score,   // 벡터 유사도 점수
           {
               etf_name: node.name,
               company: node.company,
               base_index: node.baseIndex,
               category: node.category,
               listing_date: node.listingDate,
               expense_ratio: node.expenseRatio,
               nav: node.nav,
               related_etfs: related_etfs,
               etf_code: node.code
           } AS metadata    // 메타데이터 (Langchain Document의 metadata 속성)
    """
)


# 유사도 검색 테스트
def search_etfs(query, k=5):
    """ETF 검색 함수"""
    results = vector_db.similarity_search(query, k=k)
    
    print(f"검색어: '{query}'")
    print(f"검색 결과 ({len(results)}개):")
    print("-" * 50)
    
    for i, result in enumerate(results, 1):
        metadata = result.metadata
        print(f"{i}.")
        print(f"{result.page_content}") # 페이지 내용 출력           
        print(f"상장일: {metadata.get('listing_date', 'N/A')}")
        print()


# 일반적인 카테고리 검색
search_etfs("Treasury")

Received notification from DBMS server: <GqlStatusObject gql_status='01N52', status_description='warn: unknown property key. The property `expenseRatio` does not exist. Verify that the spelling is correct.', position=<SummaryInputPosition line=38, column=36, offset=1402>, raw_classification='UNRECOGNIZED', classification=<NotificationClassification.UNRECOGNIZED: 'UNRECOGNIZED'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'UNRECOGNIZED', '_status_parameters': {'propkey': 'expenseRatio'}, '_severity': 'WARNING', '_position': {'offset': 1402, 'line': 38, 'column': 36}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: 'CALL db.index.vector.queryNodes($vector_index_name, $top_k * $effective_search_ratio, $query_vector) YIELD node, score WITH node, score LIMIT $top_k \n    // 기본 ETF 노드와 점수\n    WITH node, score\n    \n    // ETF와 관련된 모든 정보 수집\n    OPTIONAL MATCH (node)-[:BELONGS_TO]->(category:Ca

검색어: 'Treasury'
검색 결과 (5개):
--------------------------------------------------
1.
ETF 종목: TIGER 미국채10년선물
- 운용사: 미래에셋자산운용
- 기초지수: S&P 10-Year U.S. Treasury Note Futures(ER)
- 카테고리: 채권-국공채-장기
- 관련 ETF:ACE 국고채10년, ACE 미국30년국채선물레버리지(합성 H), ACE 미국30년국채액티브

상장일: 2018/08/30

2.
ETF 종목: TIGER 미국달러단기채권액티브
- 운용사: 미래에셋자산운용
- 기초지수: KIS U.S. TREASURY BOND 0-1Y 지수(총수익)
- 카테고리: 채권-국공채-단기
- 관련 ETF:ACE 단기통안채, ACE 미국달러단기채권액티브, BNK 26-06 특수채(AAA이상)액티브

상장일: 2019/07/24

3.
ETF 종목: TIGER 미국30년국채스트립액티브(합성 H)
- 운용사: 미래에셋자산운용
- 기초지수: ICE BofA Long US Treasury Principal STRIPS Index
- 카테고리: 채권-국공채-장기
- 관련 ETF:ACE 국고채10년, ACE 미국30년국채선물레버리지(합성 H), ACE 미국30년국채액티브

상장일: 2023/05/31

4.
ETF 종목: RISE 미국30년국채커버드콜(합성)
- 운용사: 케이비자산운용
- 기초지수: Bloomberg U.S. Treasury 20+ Year (TLT) 2% OTM Covered Call Index(TR)
- 카테고리: 채권-국공채-장기
- 관련 ETF:ACE 국고채10년, ACE 미국30년국채선물레버리지(합성 H), ACE 미국30년국채액티브

상장일: 2023/12/14

5.
ETF 종목: ACE 미국30년국채액티브
- 운용사: 한국투자신탁운용
- 기초지수: Bloomberg U.S Treasury 20+ Year Total Return Index
- 카테고리: 채권-국공채-장

In [68]:
# 특정 운용사 검색  
search_etfs("삼성자산 ESG")

Received notification from DBMS server: <GqlStatusObject gql_status='01N52', status_description='warn: unknown property key. The property `expenseRatio` does not exist. Verify that the spelling is correct.', position=<SummaryInputPosition line=38, column=36, offset=1402>, raw_classification='UNRECOGNIZED', classification=<NotificationClassification.UNRECOGNIZED: 'UNRECOGNIZED'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'UNRECOGNIZED', '_status_parameters': {'propkey': 'expenseRatio'}, '_severity': 'WARNING', '_position': {'offset': 1402, 'line': 38, 'column': 36}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: 'CALL db.index.vector.queryNodes($vector_index_name, $top_k * $effective_search_ratio, $query_vector) YIELD node, score WITH node, score LIMIT $top_k \n    // 기본 ETF 노드와 점수\n    WITH node, score\n    \n    // ETF와 관련된 모든 정보 수집\n    OPTIONAL MATCH (node)-[:BELONGS_TO]->(category:Ca

검색어: '삼성자산 ESG'
검색 결과 (5개):
--------------------------------------------------
1.
ETF 종목: SOL 미국S&P500ESG
- 운용사: 신한자산운용
- 기초지수: S&P500 ESG(PR)
- 카테고리: 주식-시장대표
- 관련 ETF:1Q K200액티브, 1Q 차이나H(H), 1Q 코리아밸류업

상장일: 2021/09/14

2.
ETF 종목: KODEX MSCI KOREA ESG유니버설
- 운용사: 삼성자산운용
- 기초지수: MSCI Korea ESG Universal Capped Index
- 카테고리: 주식-전략-전략테마
- 관련 ETF:ACE 2차전지&친환경차액티브, ACE ESG액티브, ACE Fn성장소비주도주

상장일: 2018/02/07

3.
ETF 종목: KODEX ESG종합채권(A-이상)액티브
- 운용사: 삼성자산운용
- 기초지수: KAP ESG 종합채권지수(A- 이상, 총수익)
- 카테고리: 채권-혼합
- 관련 ETF:HANARO 종합채권(AA-이상)액티브, KODEX 25-12 은행채(AAA)액티브

상장일: 2022/08/23

4.
ETF 종목: FOCUS ESG리더스
- 운용사: 브이아이자산운용
- 기초지수: KRX ESG Leaders 150
- 카테고리: 주식-전략-전략테마
- 관련 ETF:ACE 2차전지&친환경차액티브, ACE ESG액티브, ACE Fn성장소비주도주

상장일: 2017/12/13

5.
ETF 종목: TIGER MSCI KOREA ESG유니버설
- 운용사: 미래에셋자산운용
- 기초지수: MSCI Korea ESG Universal Index
- 카테고리: 주식-전략-전략테마
- 관련 ETF:ACE 2차전지&친환경차액티브, ACE ESG액티브, ACE Fn성장소비주도주

상장일: 2018/02/07



In [69]:
# 특정 지수 추적 ETF 검색
search_etfs("KOSPI 200 추적 ETF")

Received notification from DBMS server: <GqlStatusObject gql_status='01N52', status_description='warn: unknown property key. The property `expenseRatio` does not exist. Verify that the spelling is correct.', position=<SummaryInputPosition line=38, column=36, offset=1402>, raw_classification='UNRECOGNIZED', classification=<NotificationClassification.UNRECOGNIZED: 'UNRECOGNIZED'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'UNRECOGNIZED', '_status_parameters': {'propkey': 'expenseRatio'}, '_severity': 'WARNING', '_position': {'offset': 1402, 'line': 38, 'column': 36}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: 'CALL db.index.vector.queryNodes($vector_index_name, $top_k * $effective_search_ratio, $query_vector) YIELD node, score WITH node, score LIMIT $top_k \n    // 기본 ETF 노드와 점수\n    WITH node, score\n    \n    // ETF와 관련된 모든 정보 수집\n    OPTIONAL MATCH (node)-[:BELONGS_TO]->(category:Ca

검색어: 'KOSPI 200 추적 ETF'
검색 결과 (5개):
--------------------------------------------------
1.
ETF 종목: KOSEF 미국ETF산업STOXX
- 운용사: 키움투자자산운용
- 기초지수: STOXX USA ETF Industry Index (KRW)
- 카테고리: 주식-업종섹터-업종테마
- 관련 ETF:ACE AI반도체포커스, ACE Fn5G플러스, ACE KPOP포커스

상장일: 2022/04/26

2.
ETF 종목: KOSEF 미국S&P500(H)
- 운용사: 키움투자자산운용
- 기초지수: S&P 500
- 카테고리: 주식-시장대표
- 관련 ETF:1Q K200액티브, 1Q 차이나H(H), 1Q 코리아밸류업

상장일: 2022/12/20

3.
ETF 종목: KOSEF 미국S&P500
- 운용사: 키움투자자산운용
- 기초지수: S&P 500
- 카테고리: 주식-시장대표
- 관련 ETF:1Q K200액티브, 1Q 차이나H(H), 1Q 코리아밸류업

상장일: 2022/12/20

4.
ETF 종목: KOSEF K-테크TOP10
- 운용사: 키움투자자산운용
- 기초지수: Solactive K-TechTOP10 Index(PR)
- 카테고리: 주식-업종섹터-업종테마
- 관련 ETF:ACE AI반도체포커스, ACE Fn5G플러스, ACE KPOP포커스

상장일: 2023/10/31

5.
ETF 종목: KOSEF 글로벌전력GRID인프라
- 운용사: 키움투자자산운용
- 기초지수: NASDAQ OMX Clean Edge Smart Grid Infrastructure Index
- 카테고리: 주식-업종섹터-업종테마
- 관련 ETF:ACE AI반도체포커스, ACE Fn5G플러스, ACE KPOP포커스

상장일: 2024/08/27



In [70]:
# 배당 관련 ETF 검색
search_etfs("삼성자산 배당 ETF")

Received notification from DBMS server: <GqlStatusObject gql_status='01N52', status_description='warn: unknown property key. The property `expenseRatio` does not exist. Verify that the spelling is correct.', position=<SummaryInputPosition line=38, column=36, offset=1402>, raw_classification='UNRECOGNIZED', classification=<NotificationClassification.UNRECOGNIZED: 'UNRECOGNIZED'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'UNRECOGNIZED', '_status_parameters': {'propkey': 'expenseRatio'}, '_severity': 'WARNING', '_position': {'offset': 1402, 'line': 38, 'column': 36}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: 'CALL db.index.vector.queryNodes($vector_index_name, $top_k * $effective_search_ratio, $query_vector) YIELD node, score WITH node, score LIMIT $top_k \n    // 기본 ETF 노드와 점수\n    WITH node, score\n    \n    // ETF와 관련된 모든 정보 수집\n    OPTIONAL MATCH (node)-[:BELONGS_TO]->(category:Ca

검색어: '삼성자산 배당 ETF'
검색 결과 (5개):
--------------------------------------------------
1.
ETF 종목: KODEX MSCI EM선물(H)
- 운용사: 삼성자산운용
- 기초지수: iEdge Emerging Markets Futures Index(ER)
- 카테고리: 주식-시장대표
- 관련 ETF:1Q K200액티브, 1Q 차이나H(H), 1Q 코리아밸류업

상장일: 2018/03/23

2.
ETF 종목: KODEX 미국배당다우존스
- 운용사: 삼성자산운용
- 기초지수: Dow Jones U.S. Dividend 100 Price Return Index
- 카테고리: 주식-전략-배당
- 관련 ETF:ACE 미국배당다우존스, HANARO 고배당, KODEX 고배당

상장일: 2024/08/13

3.
ETF 종목: KODEX 미국S&P500금융
- 운용사: 삼성자산운용
- 기초지수: S&P Financial Select Sector Index(Price Return)
- 카테고리: 주식-업종섹터-금융
- 관련 ETF:KODEX 보험, KODEX 은행, KODEX 증권

상장일: 2023/03/21

4.
ETF 종목: KODEX 미국S&P500선물(H)
- 운용사: 삼성자산운용
- 기초지수: S&P 500 Futures Total Return Index
- 카테고리: 주식-시장대표
- 관련 ETF:1Q K200액티브, 1Q 차이나H(H), 1Q 코리아밸류업

상장일: 2015/05/29

5.
ETF 종목: KODEX MSCI KOREA ESG유니버설
- 운용사: 삼성자산운용
- 기초지수: MSCI Korea ESG Universal Capped Index
- 카테고리: 주식-전략-전략테마
- 관련 ETF:ACE 2차전지&친환경차액티브, ACE ESG액티브, ACE Fn성장소비주도주

상장일: 2018/02/07



## 5. Text to Cypher
- LangChain으로 Neo4J 지식 그래프 조회
- Ollama 모델 사용

`(1) GraphCypherQAChain 설정`

In [71]:
from langchain_openai import ChatOpenAI
from langchain_neo4j import GraphCypherQAChain, Neo4jGraph

# LangChain 도구 활용 - LLM 및 그래프 객체 초기화
llm = ChatOpenAI(model="gpt-4.1-mini", temperature=0.0)

graph = Neo4jGraph(
    url=os.getenv("NEO4J_URI"),
    username=os.getenv("NEO4J_USERNAME"),
    password=os.getenv("NEO4J_PASSWORD"),
    database=os.getenv("NEO4J_DATABASE"),
    enhanced_schema=True,
    refresh_schema=True,  # 스키마를 최신 상태로 유지
)

# LangChain 도구 활용 - GraphCypherQAChain 객체 초기화
cypher_chain = GraphCypherQAChain.from_llm(
    llm=llm, 
    graph=graph, 
    allow_dangerous_requests=True,  # ⚠️ 학습 목적으로만 사용. 프로덕션에서는 쿼리 검증 필요,  # ⚠️ 학습 목적으로만 사용. 프로덕션에서는 쿼리 검증 필요
    verbose=True,)

`(2) Text to Cypher - DB 조회`

In [72]:
cypher_chain.invoke({"query": "삼성자산운용의 KODEX 관련 상품은 무엇인가요?"})



> Entering new GraphCypherQAChain chain...
Generated Cypher:
MATCH (c:Company {name: "삼성자산운용"})-[:MANAGES]->(e:ETF) WHERE e.name CONTAINS "KODEX" RETURN e.code, e.name
Full Context:
[{'e.code': '069500', 'e.name': 'KODEX 200'}, {'e.code': '091160', 'e.name': 'KODEX 반도체'}, {'e.code': '091170', 'e.name': 'KODEX 은행'}, {'e.code': '091180', 'e.name': 'KODEX 자동차'}, {'e.code': '099140', 'e.name': 'KODEX 차이나H'}, {'e.code': '101280', 'e.name': 'KODEX 일본TOPIX100'}, {'e.code': '102780', 'e.name': 'KODEX 삼성그룹'}, {'e.code': '102960', 'e.name': 'KODEX 기계장비'}, {'e.code': '102970', 'e.name': 'KODEX 증권'}, {'e.code': '114260', 'e.name': 'KODEX 국고채3년'}]

> Finished chain.


{'query': '삼성자산운용의 KODEX 관련 상품은 무엇인가요?',
 'result': '삼성자산운용의 KODEX 관련 상품은 KODEX 200, KODEX 반도체, KODEX 은행, KODEX 자동차, KODEX 차이나H, KODEX 일본TOPIX100, KODEX 삼성그룹, KODEX 기계장비, KODEX 증권, KODEX 국고채3년입니다.'}

`(3) 출력 갯수를 지정 (top k)`

In [73]:
cypher_chain = GraphCypherQAChain.from_llm(
    llm=llm,
    graph=graph, 
    allow_dangerous_requests=True,  # ⚠️ 학습 목적으로만 사용. 프로덕션에서는 쿼리 검증 필요,  # ⚠️ 학습 목적으로만 사용. 프로덕션에서는 쿼리 검증 필요
    verbose=True,
    top_k=3,
)

cypher_chain.invoke({"query": "삼성자산운용의 KODEX 관련 상품은 무엇인가요?"})



> Entering new GraphCypherQAChain chain...
Generated Cypher:
MATCH (c:Company {name: "삼성자산운용"})-[:MANAGES]->(e:ETF) WHERE e.name CONTAINS "KODEX" RETURN e
Full Context:
[{'e': {'trackingError': 0.69, 'netAsset': 5604796801979, 'replicationMethod': '실물(패시브)', 'listingDate': '2002/10/14', 'code': '069500', 'baseIndex': '코스피 200', 'volatility': '높음', 'yearReturn': -5.08, 'disparityRatio': -0.04, 'totalFee': 0.15, 'name': 'KODEX 200', 'company': '삼성자산운용', 'embedding': [-0.00189971923828125, 0.0305328369140625, -0.03814697265625, 0.022186279296875, 0.006374359130859375, 0.03448486328125, -0.016571044921875, 0.017974853515625, 0.043914794921875, -0.054656982421875, 0.01027679443359375, -0.00778961181640625, -0.0039005279541015625, -0.00939178466796875, 0.0267791748046875, -0.046783447265625, 0.0008220672607421875, -0.038360595703125, -0.065185546875, -0.027374267578125, -0.0039520263671875, -0.056304931640625, 0.00250244140625, -0.03179931640625, 0.01161956787109375, -0.01061248779296875, 

{'query': '삼성자산운용의 KODEX 관련 상품은 무엇인가요?',
 'result': '삼성자산운용의 KODEX 관련 상품으로는 KODEX 200, KODEX 반도체, KODEX 은행이 있습니다.'}

`(4) 중간 결과를 포함하여 출력`

In [74]:
cypher_chain = GraphCypherQAChain.from_llm(
    llm=llm,
    graph=graph, 
    allow_dangerous_requests=True,  # ⚠️ 학습 목적으로만 사용. 프로덕션에서는 쿼리 검증 필요,  # ⚠️ 학습 목적으로만 사용. 프로덕션에서는 쿼리 검증 필요
    verbose=True,
    top_k=3,
    return_intermediate_steps=True
)

cypher_chain.invoke({"query": "삼성자산운용의 KODEX 관련 상품은 무엇인가요?"})



> Entering new GraphCypherQAChain chain...
Generated Cypher:
MATCH (c:Company {name: "삼성자산운용"})-[:MANAGES]->(e:ETF) WHERE e.name CONTAINS "KODEX" RETURN e
Full Context:
[{'e': {'trackingError': 0.69, 'netAsset': 5604796801979, 'replicationMethod': '실물(패시브)', 'listingDate': '2002/10/14', 'code': '069500', 'baseIndex': '코스피 200', 'volatility': '높음', 'yearReturn': -5.08, 'disparityRatio': -0.04, 'totalFee': 0.15, 'name': 'KODEX 200', 'company': '삼성자산운용', 'embedding': [-0.00189971923828125, 0.0305328369140625, -0.03814697265625, 0.022186279296875, 0.006374359130859375, 0.03448486328125, -0.016571044921875, 0.017974853515625, 0.043914794921875, -0.054656982421875, 0.01027679443359375, -0.00778961181640625, -0.0039005279541015625, -0.00939178466796875, 0.0267791748046875, -0.046783447265625, 0.0008220672607421875, -0.038360595703125, -0.065185546875, -0.027374267578125, -0.0039520263671875, -0.056304931640625, 0.00250244140625, -0.03179931640625, 0.01161956787109375, -0.01061248779296875, 

{'query': '삼성자산운용의 KODEX 관련 상품은 무엇인가요?',
 'result': '삼성자산운용의 KODEX 관련 상품으로는 KODEX 200, KODEX 반도체, KODEX 은행이 있습니다.',
 'intermediate_steps': [{'query': 'MATCH (c:Company {name: "삼성자산운용"})-[:MANAGES]->(e:ETF) WHERE e.name CONTAINS "KODEX" RETURN e'},
  {'context': [{'e': {'trackingError': 0.69,
      'netAsset': 5604796801979,
      'replicationMethod': '실물(패시브)',
      'listingDate': '2002/10/14',
      'code': '069500',
      'baseIndex': '코스피 200',
      'volatility': '높음',
      'yearReturn': -5.08,
      'disparityRatio': -0.04,
      'totalFee': 0.15,
      'name': 'KODEX 200',
      'company': '삼성자산운용',
      'embedding': [-0.00189971923828125,
       0.0305328369140625,
       -0.03814697265625,
       0.022186279296875,
       0.006374359130859375,
       0.03448486328125,
       -0.016571044921875,
       0.017974853515625,
       0.043914794921875,
       -0.054656982421875,
       0.01027679443359375,
       -0.00778961181640625,
       -0.0039005279541015625,
       -0.009391

`(5) cypher 쿼리 결과를 직접 출력 (LLM 답변 생성하지 않음)`

In [75]:
cypher_chain = GraphCypherQAChain.from_llm(
    llm=llm,
    graph=graph, 
    allow_dangerous_requests=True,  # ⚠️ 학습 목적으로만 사용. 프로덕션에서는 쿼리 검증 필요,  # ⚠️ 학습 목적으로만 사용. 프로덕션에서는 쿼리 검증 필요
    verbose=True,
    top_k=3,
    return_intermediate_steps=True,
    return_direct=True
)

cypher_chain.invoke({"query": "삼성자산운용의 KODEX 관련 상품은 무엇인가요?"})



> Entering new GraphCypherQAChain chain...
Generated Cypher:
MATCH (c:Company {name: "삼성자산운용"})-[:MANAGES]->(e:ETF)
WHERE e.name CONTAINS "KODEX"
RETURN e.code, e.name

> Finished chain.


{'query': '삼성자산운용의 KODEX 관련 상품은 무엇인가요?',
 'result': [{'e.code': '069500', 'e.name': 'KODEX 200'},
  {'e.code': '091160', 'e.name': 'KODEX 반도체'},
  {'e.code': '091170', 'e.name': 'KODEX 은행'}],
 'intermediate_steps': [{'query': 'MATCH (c:Company {name: "삼성자산운용"})-[:MANAGES]->(e:ETF)\nWHERE e.name CONTAINS "KODEX"\nRETURN e.code, e.name'}]}

`(6) cypher 쿼리 생성하는 모델과 최종 답변 생성 모델을 별도 적용`

In [78]:
from langchain_ollama import ChatOllama

second_llm = ChatOllama(model="gemma3", temperature=0.0)

cypher_chain = GraphCypherQAChain.from_llm(
    cypher_llm=llm,
    qa_llm=second_llm,
    graph=graph, 
    allow_dangerous_requests=True,  # ⚠️ 학습 목적으로만 사용. 프로덕션에서는 쿼리 검증 필요,  # ⚠️ 학습 목적으로만 사용. 프로덕션에서는 쿼리 검증 필요
    verbose=True,
)

cypher_chain.invoke({"query": "삼성자산운용의 KODEX 관련 상품은 무엇인가요?"})



> Entering new GraphCypherQAChain chain...
Generated Cypher:
MATCH (c:Company {name: "삼성자산운용"})-[:MANAGES]->(e:ETF)
WHERE e.name CONTAINS "KODEX"
RETURN e.code, e.name
Full Context:
[{'e.code': '069500', 'e.name': 'KODEX 200'}, {'e.code': '091160', 'e.name': 'KODEX 반도체'}, {'e.code': '091170', 'e.name': 'KODEX 은행'}, {'e.code': '091180', 'e.name': 'KODEX 자동차'}, {'e.code': '099140', 'e.name': 'KODEX 차이나H'}, {'e.code': '101280', 'e.name': 'KODEX 일본TOPIX100'}, {'e.code': '102780', 'e.name': 'KODEX 삼성그룹'}, {'e.code': '102960', 'e.name': 'KODEX 기계장비'}, {'e.code': '102970', 'e.name': 'KODEX 증권'}, {'e.code': '114260', 'e.name': 'KODEX 국고채3년'}]


ResponseError: model 'gemma3' not found (status code: 404)

`(7) 사용자 프롬프트를 직접 지정`

In [ ]:
print(graph.schema)

In [ ]:
from langchain_core.prompts.prompt import PromptTemplate

# Cypher 생성을 위한 프롬프트
CYPHER_GENERATION_TEMPLATE = """Task: Generate Cypher statement to query a graph database.
Instructions:
Use only the provided relationship types and properties in the schema.
Do not use any other relationship types or properties that are not provided.

<Schema>
{schema}
</Schema>

<Note>
Do not include any explanations or apologies in your responses.
Do not respond to any questions that might ask anything else than for you to construct a Cypher statement.
Do not include any text except the generated Cypher statement.
</Note>

######################
#  Here are a few examples of generated Cypher statements for particular questions:
######################

<Examples>
Q1. 수익률이 가장 높은 5개 ETF는 무엇인가요?
MATCH (e:ETF)
WHERE e.yearReturn IS NOT NULL
RETURN e.name, e.code, e.yearReturn
ORDER BY e.yearReturn DESC
LIMIT 5

Q2. 미래에셋자산운용에서 운용하는 모든 ETF를 보여주세요.
MATCH (c:Company {{name: '미래에셋자산운용'}})-[:MANAGES]->(e:ETF)
RETURN e.name, e.code, e.category, e.yearReturn
ORDER BY e.yearReturn DESC

Q3. 국내 주식형 ETF 중 순자산총액이 가장 큰 상위 10개는 무엇인가요?
MATCH (c:Category {{name: '국내 주식형'}})<-[:BELONGS_TO]-(e:ETF)
WHERE e.netAsset IS NOT NULL
RETURN e.name, e.code, e.netAsset, e.yearReturn
ORDER BY e.netAsset DESC
LIMIT 10

Q3. 순자산총액이 1조원 이상인 ETF의 평균 수익률은 얼마인가요?
MATCH (e:ETF)
WHERE e.netAsset >= 1000000000000 AND e.yearReturn IS NOT NULL
RETURN AVG(e.yearReturn) AS averageReturn, COUNT(e) AS etfCount

Q4. 운용사별 평균 ETF 수익률은 어떻게 되나요?
MATCH (c:Company)-[:MANAGES]->(e:ETF)
WHERE e.yearReturn IS NOT NULL
WITH c.name AS company, AVG(e.yearReturn) AS avgReturn, COUNT(e) AS etfCount
ORDER BY avgReturn DESC
RETURN company, avgReturn, etfCount
</Examples>

The question is:
{question}"""

# 결과 처리를 위한 QA 프롬프트
QA_TEMPLATE = """
You are a financial assistant providing information about ETF databases in an easy-to-understand manner in Korean.
Based on the information obtained from the ETF database, answer the question in a clear and informative way.

Question: {question}
Search Results: {context}

Respond only with relevant information in a natural, conversational tone. Convert numerical data such as returns and net asset values into appropriate units (%, million USD, billion USD, etc.) to make them easily readable.
Avoid content that could be interpreted as investment advice or recommendations, and only convey objective facts.
"""

CYPHER_GENERATION_PROMPT = PromptTemplate(
    input_variables=["schema", "question"], 
    template=CYPHER_GENERATION_TEMPLATE
)

QA_PROMPT = PromptTemplate(
    input_variables=["question", "context"], 
    template=QA_TEMPLATE
)

# Chain 생성 - input_key와 output_key를 명시적으로 설정
cypher_chain = GraphCypherQAChain.from_llm(
    cypher_llm=llm,
    qa_llm=second_llm,
    graph=graph, 
    allow_dangerous_requests=True,  # ⚠️ 학습 목적으로만 사용. 프로덕션에서는 쿼리 검증 필요,  # ⚠️ 학습 목적으로만 사용. 프로덕션에서는 쿼리 검증 필요
    verbose=True,
    cypher_prompt=CYPHER_GENERATION_PROMPT,
    qa_prompt=QA_PROMPT,
    input_key="question",  
    output_key="result"
)

# 쿼리 실행 - 입력 키를 "question"으로 변경
cypher_chain.invoke({"question": "삼성자산운용의 KODEX 관련 상품은 무엇인가요?"})

## 6. 벡터 검색(Semantic Search) 

> 💡 **Vector Search 파이프라인 한 줄 요약**
>
> `텍스트 → OpenAI 임베딩 → Neo4j 노드에 저장 → VECTOR INDEX → similarity_search`
>
> 🔑 **핵심 포인트**: Neo4jVector 는 별도 벡터 DB 가 아니라 **기존 노드의 속성**(`embedding`)에 
> 임베딩을 저장하는 방식입니다. 그래서 "이 ETF와 비슷한 ETF" 같은 질의를 
> **그래프 탐색 + 벡터 검색** 을 하나의 Cypher 로 섞어 쓸 수 있습니다.


`(1) Graph DB를 초기화`

In [ ]:
from langchain_openai import OpenAIEmbeddings
from langchain_neo4j import Neo4jVector

embeddings = OpenAIEmbeddings(model="text-embedding-3-small") 

existing_graph = Neo4jVector.from_existing_index(
    embeddings,
    url=NEO4J_URI,
    username=NEO4J_USERNAME,
    password=NEO4J_PASSWORD,
    index_name="etf_vector_index",  # 기존에 생성한 벡터 인덱스 이름
    node_label="ETF",  # 노드 레이블
    text_node_property="name",  # 텍스트 속성 (ETF 이름)
    embedding_node_property="embedding",  # 임베딩 속성
    retrieval_query="""
    WITH node, score
    WITH node, score,
         "ETF 종목: " + node.name + 
         CASE WHEN node.company IS NOT NULL THEN "\n- 운용사: " + node.company ELSE "" END +
         CASE WHEN node.baseIndex IS NOT NULL THEN "\n- 기초지수: " + node.baseIndex ELSE "" END +
         CASE WHEN node.category IS NOT NULL THEN "\n- 카테고리: " + node.category ELSE "" END AS combined_text
         
    RETURN combined_text AS text,   // 텍스트 (LangChain Document 객체의 page_content 속성)
           score,  // 유사도 점수 
           {
               etf_name: node.name,
               company: node.company,
               base_index: node.baseIndex,
               category: node.category,
               listing_date: node.listingDate,
               total_fee: node.totalFee,
               etf_code: node.code
           } AS metadata    // 메타데이터 (LangChain Document 객체의 metadata 속성)
    """
)

# 벡터 검색 쿼리 실행
query = "삼성자산운용의 KRW 관련 상품은 무엇인가요?" # 검색할 쿼리
similar_docs = existing_graph.similarity_search_with_score(query, k=5) # 유사도 상위 5개 문서 검색

for doc, score in similar_docs:
    print(f"문서 내용: {doc.page_content}, 유사도 점수: {score}")

`(2) Graph DB 검색을 수행하는 함수`

In [ ]:
# Cypher 쿼리 실행 및 관련 ETF 노드 검색을 처리하는 함수 
def execute_query_and_get_etf_data(query, k=5):
    # 벡터 검색 쿼리 실행
    similar_docs = existing_graph.similarity_search_with_score(query, k) # 유사도 상위 k개 문서 검색
    similar_doc_names = [doc.metadata['etf_name'] for doc, _ in similar_docs] # 유사한 문서의 종목명 추출

    # 유사한 문서의 종목명을 기반으로 ETF 노드 검색
    etf_query = """
    MATCH (e:ETF)   // 모든 ETF 노드 검색
    WHERE e.name IN $similar_doc_names   // 유사한 문서의 종목명과 일치하는 ETF 노드 검색
    RETURN e
    """
    etf_results = graph.query(
        etf_query, 
        params={"similar_doc_names": similar_doc_names}
    ) # ETF 노드 검색
    etf_results = [etf['e'] for etf in etf_results] # 검색된 ETF 노드 반환

    # 문자열 포맷팅
    result = ""
    for etf in etf_results:
        result += f"# 종목명: {etf['name']} (운용사: {etf['company']})\n"
        result += f"- 기초지수: {etf['baseIndex']}, 수익률: {etf['yearReturn']}, 순자산총액: {etf['netAsset']}\n"
        result += f"- 추적오차: {etf['trackingError']}, 변동성: {etf['volatility']}, 복제방법: {etf['replicationMethod']}, 총보수: {etf['totalFee']}, 과세유형: {etf['taxType']}\n"
        result += "-" * 3 + "\n"
    return result


# 쿼리 실행 및 관련 ETF 노드 검색
query = "삼성자산운용의 KODEX 관련 상품은 무엇인가요?" 
etf_data = execute_query_and_get_etf_data(query, k=5) # ETF 노드 검색
print(etf_data)

`(3) LCEL 사용하여 RAG 체인을 구성`

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough, RunnableLambda

# LLM
llm = ChatOpenAI(model="gpt-4.1-mini", temperature=0.0)

# Prompt
template = '''당신은 ETF 데이터 분석 전문가로서 오직 주어진 정보에 기반하여 객관적이고 정확한 답변을 제공합니다.

주어진 정보:
{context}

질문: {question}

답변 작성 지침:
1. 제공된 정보에 명시된 사실만 사용하세요.
2. 수익률은 '%', 순자산총액은 '억 원' 또는 '조 원' 단위로 표시하세요.
3. 투자 조언이나 추천으로 해석될 수 있는 표현은 사용하지 마세요.
4. 제공된 정보에 없는 내용은 "제공된 정보에서 해당 내용을 찾을 수 없습니다"라고 답하세요.
5. 한국어로 자연스럽고 이해하기 쉽게 답변하세요.
'''

prompt = ChatPromptTemplate.from_template(template)

# RAG Chain 연결
rag_chain = (
    {'context': RunnableLambda(execute_query_and_get_etf_data), 'question': RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

# Chain 실행
query = "삼성자산운용의 KODEX 관련 상품은 무엇인가요?" 
answer = rag_chain.invoke(query)

print(answer)

### **[실습]**

S&P 500 회사 데이터(data/sp500_companies.csv)를 활용하여 Knowledge Graph를 구현합니다.

#### 실습 요구사항

**1. 데이터 모델링**
- 노드 타입: Company, Sector, Industry, Exchange
- 관계 구조: `(Company)-[:BELONGS_TO]->(Sector)-[:PART_OF]->(Industry)`
- 추가 관계: `(Company)-[:LISTED_ON]->(Exchange)`

**2. 그래프 구축**
- CSV 데이터 로드 및 전처리 (결측값 처리 포함)
- 노드 생성 시 UNIQUE 제약조건 적용
- 관계 생성 및 검증

**3. 검색 기법 구현**
다음 기법 중 최소 3가지를 구현하세요:
- [ ] **Text-to-Cypher**: LLM을 활용한 자연어 쿼리 변환
- [ ] **Full-text Search**: 회사명 기반 전문 검색 인덱스
- [ ] **Vector Search**: 회사 설명 임베딩 및 유사도 검색
- [ ] **Graph Pattern Matching**: Sector/Industry 기반 연관 회사 탐색
- [ ] **Hybrid Search**: 벡터 검색 + Cypher 필터링 조합

**4. 검증 쿼리 예시**
- "Technology Sector에 속한 회사는 몇 개인가요?"
- "Apple과 같은 Industry의 회사를 찾아주세요"
- "NASDAQ에 상장된 회사 중 시가총액 상위 10개"

**출처**: https://www.kaggle.com/datasets/andrewmvd/sp-500-stocks?resource=download

#### 1. 데이터 로드 및 전처리

In [ ]:
sp500_df = pd.read_csv("./data/sp500_companies.csv")
sp500_df.head()

#### 2. 제약 조건 설정

In [ ]:
# 여기에 코드를 작성하세요.


<details close>
<summary>💡 정답 확인</summary>

```python
# UNIQUE 제약조건 생성
constraints = [
    "CREATE CONSTRAINT company_symbol_unique IF NOT EXISTS FOR (c:Company) REQUIRE c.symbol IS UNIQUE",
    "CREATE CONSTRAINT sector_name_unique IF NOT EXISTS FOR (s:Sector) REQUIRE s.name IS UNIQUE",
    "CREATE CONSTRAINT industry_name_unique IF NOT EXISTS FOR (i:Industry) REQUIRE i.name IS UNIQUE",
    "CREATE CONSTRAINT exchange_name_unique IF NOT EXISTS FOR (e:Exchange) REQUIRE e.name IS UNIQUE"
]

for constraint in constraints:
    try:
        graph.query(constraint)
        print(f"제약조건 생성 완료: {constraint.split('FOR')[1].split('REQUIRE')[0].strip()}")
    except Exception as e:
        print(f"제약조건 생성 실패 또는 이미 존재: {e}")
```

</details>

#### 3. 노드 및 관계 생성

In [ ]:
# 여기에 코드를 작성하세요.


<details close>
<summary>💡 정답 확인</summary>

```python
# 결측값 처리
sp500_df['Sector'] = sp500_df['Sector'].fillna('Unknown')
sp500_df['Industry'] = sp500_df['Industry'].fillna('Unknown')
sp500_df['Exchange'] = sp500_df['Exchange'].fillna('Unknown')

# 노드 및 관계 생성 (배치 처리)
for _, row in sp500_df.iterrows():
    cypher_query = """
    // Exchange 노드 생성
    MERGE (e:Exchange {name: $exchange})
    
    // Sector 노드 생성
    MERGE (s:Sector {name: $sector})
    
    // Industry 노드 생성
    MERGE (i:Industry {name: $industry})
    
    // Company 노드 생성 및 속성 설정
    MERGE (c:Company {symbol: $symbol})
    SET c.shortname = $shortname,
        c.longname = $longname,
        c.currentprice = $currentprice,
        c.marketcap = $marketcap,
        c.description = $description
    
    // 관계 생성
    MERGE (c)-[:BELONGS_TO]->(s)
    MERGE (s)-[:PART_OF]->(i)
    MERGE (c)-[:LISTED_ON]->(e)
    """
    
    params = {
        "symbol": row['Symbol'],
        "shortname": row['Shortname'],
        "longname": row['Longname'],
        "sector": row['Sector'],
        "industry": row['Industry'],
        "exchange": row['Exchange'],
        "currentprice": float(row['Currentprice']) if pd.notna(row['Currentprice']) else 0.0,
        "marketcap": float(row['Marketcap']) if pd.notna(row['Marketcap']) else 0.0,
        "description": str(row['Longbusinesssummary']) if pd.notna(row['Longbusinesssummary']) else ""
    }
    
    try:
        graph.query(cypher_query, params=params)
    except Exception as e:
        print(f"노드 생성 실패 ({row['Symbol']}): {e}")

print("노드 및 관계 생성 완료!")

# 생성 확인
result = graph.query("""
    MATCH (c:Company)
    RETURN count(c) as company_count
""")
print(f"생성된 회사 노드 수: {result[0]['company_count']}")
```

</details>

#### 4. 데이터 확인 및 검색

In [ ]:
# 여기에 코드를 작성하세요.


<details close>
<summary>💡 정답 확인</summary>

```python
# 1. Technology Sector에 속한 회사 개수
result = graph.query("""
    MATCH (c:Company)-[:BELONGS_TO]->(s:Sector {name: 'Technology'})
    RETURN count(c) as count
""")
print(f"Technology Sector 회사 수: {result[0]['count']}")

# 2. Apple과 같은 Industry의 회사 찾기
result = graph.query("""
    MATCH (apple:Company {symbol: 'AAPL'})-[:BELONGS_TO]->(s:Sector)-[:PART_OF]->(i:Industry)
    MATCH (other:Company)-[:BELONGS_TO]->(s)-[:PART_OF]->(i)
    WHERE other.symbol <> 'AAPL'
    RETURN other.shortname as company, other.symbol as symbol
    LIMIT 10
""")
print("\nApple과 같은 Industry의 회사:")
for record in result:
    print(f"  - {record['company']} ({record['symbol']})")

# 3. NASDAQ에 상장된 회사 중 시가총액 상위 10개
result = graph.query("""
    MATCH (c:Company)-[:LISTED_ON]->(e:Exchange)
    WHERE e.name CONTAINS 'NMS' OR e.name CONTAINS 'NASDAQ'
    RETURN c.shortname as company, c.symbol as symbol, c.marketcap as marketcap
    ORDER BY c.marketcap DESC
    LIMIT 10
""")
print("\nNASDAQ 시가총액 상위 10개:")
for i, record in enumerate(result, 1):
    marketcap_b = record['marketcap'] / 1e9 if record['marketcap'] else 0
    print(f"  {i}. {record['company']} ({record['symbol']}): ${marketcap_b:.2f}B")
```

</details>

#### 5. 벡터 인덱스 생성

In [ ]:
# 여기에 코드를 작성하세요.


<details close>
<summary>💡 정답 확인</summary>

```python
from langchain_openai import OpenAIEmbeddings
from langchain_neo4j import Neo4jVector
from langchain_core.documents import Document

# 임베딩 모델 초기화
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

# Company 노드에서 description 가져오기
companies_result = graph.query("""
    MATCH (c:Company)
    WHERE c.description IS NOT NULL AND c.description <> ''
    RETURN c.symbol as symbol, c.shortname as name, c.description as description
    LIMIT 100
""")

# LangChain Document 형식으로 변환
docs = [
    Document(
        page_content=company['description'],
        metadata={
            'symbol': company['symbol'],
            'name': company['name']
        }
    )
    for company in companies_result
]

# Neo4j 벡터 스토어 생성
vector_store = Neo4jVector.from_documents(
    docs,
    embeddings,
    url=os.getenv("NEO4J_URI"),
    username=os.getenv("NEO4J_USERNAME"),
    password=os.getenv("NEO4J_PASSWORD"),
    index_name="sp500_vector_index",
    node_label="Company",
    text_node_property="description",
    embedding_node_property="embedding"
)

print(f"벡터 인덱스 생성 완료! ({len(docs)}개 회사)")

# 벡터 검색 테스트
query = "AI and machine learning technology company"
results = vector_store.similarity_search(query, k=5)

print(f"\n검색 쿼리: {query}")
print("유사한 회사:")
for i, doc in enumerate(results, 1):
    print(f"  {i}. {doc.metadata['name']} ({doc.metadata['symbol']})")
```

</details>